# Readme

In [ ]:
task = ["price","rent"]    # price=sell, rent= rent
str0 = ["training","exam"]
debug_mode = False
silent = True
verbose = not silent
# Python version: 3.12.8
# Change debug_mode and silent to view more output or less
# If debug mode is on, function will display analysis function

output score:  
for price:  
for rent:  
kaggle:
OLS - 59.9  
LASSO - 58.2  
Ridge - 59.9  
ElasticNet - 59.0
Afterwards, function will output kaggle score as well, but it's not truth.

# utility function

## preparation for environment

In [2]:
import os
import warnings
import pandas as pd
import numpy as np
import sys
from scipy import stats
from scipy.stats import pearsonr, skew
import re
from sklearn.neighbors import KNeighborsClassifier
from collections import Counter
from typing import Tuple, List, Optional, NoReturn, Union, Any, List, Dict

## robust load data

There are three functions: find_file, robust_file_finder, robust_read_csv  
In main function,                we use robust_read_csv and robust_file_finder to make sure data is loaded successfully

In [3]:
def find_file(substring1, substring2=None, directory="."):
    """
    This function is used to handle issue if expected file cannot be found.
    Find files in the working directory that contain specified substrings.
    
    Parameters:
    -----------
    substring1 : str
        First substring that must be contained in filename
    substring2 : str, optional
        Second substring that must be contained in filename (optional)
    directory : str
        Search directory, defaults to current directory
    
    Returns:
    --------
    str or None
        Found filename, returns None if not found
    """
    # Check if directory exists
    if not os.path.exists(directory):
        warnings.warn(f"Directory '{directory}' does not exist")
        return None
    
    # Get all files in directory
    try:
        all_files = os.listdir(directory)
    except PermissionError:
        warnings.warn(f"No permission to access directory '{directory}'")
        return None
    
    # Filter files that match criteria
    matching_files = []
    
    for filename in all_files:
        # smartly combine directory with file
        file_path = os.path.join(directory, filename)
        
        # Skip directories, only consider files
        if not os.path.isfile(file_path):
            continue
            
        # Skip non-CSV/TXT files
        if not (filename.lower().endswith('.csv') or filename.lower().endswith('.txt')):
            continue

        # Check if contains substrings (case-insensitive)
        filename_lower = filename.lower()
        substring1_lower = substring1.lower()
        
        contains_sub1 = substring1_lower in filename_lower
        contains_sub2 = substring2 is None or (substring2.lower() in filename_lower)
        
        if contains_sub1 and contains_sub2:
            matching_files.append(filename)
    
    # Handle results
    if len(matching_files) == 0:
        return None
    elif len(matching_files) == 1:
        return matching_files[0]
    else:
        # If multiple files found, return first one with warning
        warnings.warn(f"Found multiple matching files: {matching_files}, using first: {matching_files[0]}")
        return matching_files[0]



def robust_file_finder(target_filename, directory="."):
    """
    Robustly find files using exact match first, then fuzzy match.
    
    Parameters:
    -----------
    target_filename : str
        Target filename to find
    directory : str
        Directory to search in
    
    Returns:
    --------
    str
        Found file path
    
    Raises:
    -------
    FileNotFoundError
        If file cannot be found
    """
    file_path = os.path.join(directory, target_filename)
    
    # 1. First try exact match
    if os.path.exists(file_path):
        print(f"Success: exact match found: {target_filename}")
        return file_path
    
    # 2. If exact match fails, try fuzzy matching
    print(f"Warning: exact match not found for: {target_filename}, starting fuzzy search...")
    
    # Determine search substrings based on filename pattern
    target_lower = target_filename.lower()
    str1 = None
    str2 = None
    # Set first substring based on data type
    if "test" in target_lower:
        str1 = "test"
    elif "train" in target_lower:
        str1 = "train"
    
    # Set second substring based on prediction target
    if "price" in target_lower:
        str2 = "price"
    elif "rent" in target_lower:
        str2 = "rent"
    
    if str1 is None or str2 is None:
        error_msg = f"Cannot determine search pattern for: {target_filename} (missing 'test/train' or 'price/rent')"
        raise FileNotFoundError(error_msg)
    
    # fussy search
    found_file = find_file(str1, str2, directory)
    
    if found_file:
        found_path = os.path.join(directory, found_file)
        print(f"Fuzzy match found: {found_file} (original target: {target_filename})")
        return found_path
    else:
        error_msg = f"Cannot find file: {target_filename} (tried exact and fuzzy matching)"
        raise FileNotFoundError(error_msg)




def robust_read_csv(target_filename, directory=".", encoding_options=None):
    """
    Robustly read CSV file with multiple encoding attempts.
    Handles UTF-8 BOM and other common encodings for Chinese text.
    """
    import pandas as pd
    if encoding_options is None:
        encoding_options = ['utf-8-sig', 'utf-8', 'gbk', 'gb2312', 'latin1']
    
    # Find file using robust finder
    file_path = robust_file_finder(target_filename, directory)
    
    # Try different encodings
    last_error = None
    for encoding in encoding_options:
        try:
            print(f"Trying encoding: {encoding}")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"Successfully read {target_filename} with {encoding} encoding")
            return df
        except UnicodeDecodeError as e:
            last_error = e
            print(f"  {encoding} failed: {e}")
            continue
        except Exception as e:
            last_error = e
            print(f"  {encoding} failed: {e}")
            continue
    
    # If all encodings fail
    error_msg = f"Failed to read {target_filename} with encodings: {encoding_options}"
    raise ValueError(error_msg) from last_error

def final_load_data(main_index, task):
    # Initialize variables to None
    data = None
    exam = None
    
    # load data for exam
    try:
        predict_path = robust_file_finder(f"ruc_Class25Q2_test_{task[main_index]}.csv")
        exam = robust_read_csv(predict_path)
        print(f"Success: find {task[main_index]} for test")
    except FileNotFoundError as e:
        print(f"Failed: {e}")
        print(f"Error: fail to find {task[main_index]} for test")

    # load data for training
    try:
        data_path = robust_file_finder(f"ruc_Class25Q2_train_{task[main_index]}.csv")
        data = robust_read_csv(data_path)
        print(f"Success: find {task[main_index]} for train")
    except FileNotFoundError as e:
        print(f"Failed: {e}")
        print(f"Error: fail to find {task[main_index]} for train")

    # check whether data is loaded successfully
    if data is not None and exam is not None:
        print("Data is ready!")
    else:
        print("Check whether file exists in working directory.")
        print("Program exit.")
        sys.exit(1)
    
    return data, exam

## data analysis

### convert to set

In [4]:
def transfer_to_set(data: pd.DataFrame, size: Optional[int] = None) -> Tuple[List[str], List[set]]:
    """
    Convert each column of DataFrame to actual set objects.
    """
    column_list = []
    set_list = []
    
    for column in data.columns:
        column_list.append(column)
        
        # Get unique values from the column
        unique_values = data[column].dropna().unique()
        
        # Limit the number of values if size is specified
        if size is not None and len(unique_values) > size:
            sample_values = unique_values[:size]
        else:
            sample_values = unique_values
        
        # Create actual set object
        value_set = set()
        for value in sample_values:
            if pd.isna(value) or value == "":
                continue
            value_set.add(value)
            
        set_list.append(value_set)
    
    return column_list, set_list

### missing report

In [5]:
def data_missing_report(data: pd.DataFrame,display_row :bool = True, display_column :bool = True):
    """
    Generate a comprehensive report on missing values in DataFrame.
    
    Args:
        data: Input pandas DataFrame to analyze
    """
    total_rows = len(data)
    total_columns = len(data.columns)
    if display_row == True:
        # Row statistics
        print("\n--- ROW STATISTICS ---")
        
        # Calculate missing values per row
        missing_per_row = data.isnull().sum(axis=1)
        
        
        print(f"Total samples: {total_rows}")
        print(f"Total indicators: {total_columns}")
        
        # Count rows with different numbers of missing values
        for missing_count in range(total_columns + 1):
            count = (missing_per_row == missing_count).sum()
            percentage = (count / total_rows) * 100 if total_rows > 0 else 0
            
            print(f"Samples missing {missing_count} indicators: count = {count}, percentage = {percentage:.2f}%")
    
    if display_column == True:
        # Column statistics - sorted by missing count descending
        print("\n--- COLUMN STATISTICS (Sorted by missing count descending) ---")
        print(f"Total samples: {total_rows}")
        
        # Calculate missing counts for all columns and sort
        missing_stats = []
        for i, column in enumerate(data.columns):
            missing_count = data[column].isnull().sum()
            percentage = (missing_count / total_rows) * 100 if total_rows > 0 else 0
            missing_stats.append({
                'original_index': i + 1,
                'column_name': column,
                'missing_count': missing_count,
                'missing_percentage': percentage
            })
        
        # Sort by missing count descending
        missing_stats.sort(key=lambda x: x['missing_count'], reverse=True)
        
        # Print sorted results
        for stat in missing_stats:
            print(f"Column {stat['original_index']} name: {stat['column_name']}, "
                f"missing count = {stat['missing_count']}, "
                f"percentage = {stat['missing_percentage']:.2f}%")

### linear_correlation and vif

In [6]:
def linear_correlation_analysis(
    df: pd.DataFrame,
    col1: str,
    col2: str,
    threshold: float = 0.8,
    significance_level: float = 0.05,
    silent: bool = False
) -> Tuple[int, float]:
    """
    Analyze correlation between two numerical columns.
    Returns (status, pearson_r)
      status:
        1  = strong correlation confirmed (both Pearson and Spearman |r|>threshold and p<significance_level)
        0  = weak or non-significant
       -1  = invalid (missing cols, not numeric, insufficient data, error)
      pearson_r: Pearson correlation coefficient (or np.nan if invalid)
    If silent==False, prints a short report.
    """
    def log(*args, **kwargs):
        if not silent:
            print(*args, **kwargs)

    # make sure data exist
    if col1 not in df.columns or col2 not in df.columns:
        log(f"Error: Columns '{col1}' or '{col2}' not found")
        return -1, np.nan

    complete = df[df[col1].notna() & df[col2].notna()]
    if len(complete) < 2:
        log("Error: Insufficient complete data (need at least 2 samples)")
        return -1, np.nan

    if not (pd.api.types.is_numeric_dtype(complete[col1]) and pd.api.types.is_numeric_dtype(complete[col2])):
        log(f"Error: Columns '{col1}' and '{col2}' must be numeric")
        return -1, np.nan

    x = complete[col1]
    y = complete[col2]
    try:
        # transfer to float
        pearson_r, pearson_p = map(float, pearsonr(x, y))
    except Exception as e:
        log(f"Error computing correlations: {e}")
        return -1, np.nan

    if not silent:
        print(f"=== {col1} vs {col2} ===")
        print(f"  samples: {len(complete)}")
        print(f"  Pearson:  r={pearson_r:.6f}, p={pearson_p:.6g}")

    if abs(pearson_r) > threshold and (pearson_p < significance_level):
        if not silent:
            print("  => overall: STRONG")
        return 1, pearson_r
    else:
        if not silent:
            print("  => overall: weak / non-significant")
        return 0, pearson_r

# Assumes linear_correlation_analysis(df, col1, col2, threshold, significance_level, silent)
# is available in the namespace and returns Tuple[int, float] as you specified.

def choose_drop_for_pair(
    df: pd.DataFrame,
    a: str,
    b: str,
    missing_rate_threshold: float = 0.10,
    var_tol: float = 1e-12
) -> str:
    """
    Decide which column to drop among (a, b) using the rule:
      - If |a_na - b_na| > missing_rate_threshold: drop the column with higher missing rate.
      - Else drop the column with smaller variance (population var ddof=0).
      - Deterministic fallback: drop the one with higher missing rate, then lexicographic.
    Returns the chosen column name to drop (always returns a string).
    """
    a_na = float(df[a].isna().mean()) if a in df.columns else 1.0
    b_na = float(df[b].isna().mean()) if b in df.columns else 1.0

    if abs(a_na - b_na) > missing_rate_threshold:
        return a if a_na > b_na else b

    a_vals = df[a].dropna()
    b_vals = df[b].dropna()

    def safe_var(s: pd.Series) -> float:
        if s.size == 0:
            return float('nan')
        try:
            return float(s.var(ddof=0))
        except Exception:
            return float('nan')

    a_var = safe_var(a_vals)
    b_var = safe_var(b_vals)

    # both nan -> fallback to missing rate (they are close)
    if np.isnan(a_var) and np.isnan(b_var):
        if a_na != b_na:
            return a if a_na > b_na else b
        return min(a, b)

    # one nan -> drop that one
    if np.isnan(a_var):
        return a
    if np.isnan(b_var):
        return b

    # drop smaller variance
    if a_var + var_tol < b_var:
        return a
    if b_var + var_tol < a_var:
        return b

    # tie -> drop higher missing
    if a_na != b_na:
        return a if a_na > b_na else b

    # final deterministic fallback
    return min(a, b)


def VIF_process(
    data: pd.DataFrame,
    exam: pd.DataFrame,
    linear_correlation_threshold: float = 0.80,
    p_value: float = 0.05,
    label_str: str = 'ID',
    y_str: str = 'Price',
    perform_delete: bool = True,
    missing_rate_threshold: float = 0.10,
    silent: bool = False
) -> Tuple[Optional[pd.DataFrame], List[str], List[Tuple[str, str, float]]]:
    """
    VIF_process (correlation-first variant).
    - Computes correlation matrix over numeric/bool columns (excluding label_str and y_str).
    - Prints a notification once the correlation matrix is computed.
    - Finds candidate pairs with |r| >= linear_correlation_threshold.
    - For each candidate pair, calls linear_correlation_analysis(...) to confirm significance.
      If confirmed, decides which column to drop using `choose_drop_for_pair`.
    - If perform_delete==True then drops the chosen columns in-place from both `data` and `exam`.
    Returns:
      (corr_df_or_None, dropped_columns_list, confirmed_pairs_list)
        - corr_df_or_None: the computed correlation DataFrame (or None if it couldn't be built)
        - dropped_columns_list: list of columns actually dropped (may be empty)
        - confirmed_pairs_list: list of (colA, colB, pearson_r) that were confirmed by statistical test
    Notes:
      - This function calls `linear_correlation_analysis` from the surrounding namespace.
      - Deletion is done IN-PLACE on the provided DataFrames when perform_delete==True.
    """
    # step: prepare training frame (do not alter original until deletion)
    df_train = data
    # drop label and y if present for purposes of correlation analysis
    analysis_cols = [c for c in df_train.columns if c not in {label_str, y_str}]
    # select numeric and boolean columns only
    df_num = df_train[analysis_cols].select_dtypes(include=[np.number, 'bool']).copy()

    if df_num.shape[1] == 0:
        if not silent:
            print("VIF_process: no numeric/bool columns found for correlation/VIF analysis.")
        return None, [], []

    # compute correlation matrix (pandas .corr handles pairwise complete cases by default)
    try:
        corr_df = df_num.corr()
    except Exception as e:
        if not silent:
            print(f"VIF_process: failed to compute correlation matrix: {e}")
        return None, [], []

    # notify that correlation matrix is ready
    if not silent:
        print("Correlation matrix computed.")

    # sanity check: any NaN/Inf in correlation matrix -> return early with the corr_df
    if corr_df.isna().values.any() or np.isinf(corr_df.to_numpy()).any():
        if not silent:
            print("VIF_process: correlation matrix contains NaN or Inf; exporting and stopping.")
        return corr_df, [], []

    cols = corr_df.columns.tolist()

    candidate_pairs: List[Tuple[str, str, float]] = []
    confirmed_pairs: List[Tuple[str, str, float]] = []
    to_drop: List[str] = []

    # collect candidate pairs by threshold
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            a = cols[i]
            b = cols[j]
            r = corr_df.iat[i, j]
            if pd.isna(r):
                continue
            if abs(r) >= linear_correlation_threshold:
                candidate_pairs.append((a, b, float(r)))

    if not silent:
        print(f"VIF_process: found {len(candidate_pairs)} candidate pairs with |r| >= {linear_correlation_threshold:.3f}.")

    # ensure linear_correlation_analysis is available
    lca = globals().get('linear_correlation_analysis', None)
    if not callable(lca):
        if not silent:
            print("VIF_process: linear_correlation_analysis not found in global scope. Aborting pair confirmation.")
        return corr_df, [], candidate_pairs

    # confirm with statistical test and decide drop
    for a, b, r in candidate_pairs:
        try:
            status, pearson_r = lca(df_num, a, b, threshold=linear_correlation_threshold,
                                    significance_level=p_value, silent=True)
        except Exception as e:
            # if the linear test raises, skip
            if not silent:
                print(f"VIF_process: linear test error for ({a},{b}): {e}; skipping.")
            continue

        if status != 1:
            # not confirmed by statistical test
            continue

        # confirmed strong pair -> decide which to drop
        pick = choose_drop_for_pair(df_num, a, b, missing_rate_threshold=missing_rate_threshold)
        if pick not in to_drop:
            to_drop.append(pick)
        confirmed_pairs.append((a, b, float(pearson_r)))

        if not silent:
            a_na = float(df_num[a].isna().mean())
            b_na = float(df_num[b].isna().mean())
            print(f"  Confirmed strong pair: ({a}, {b}), r={r:.6f}. Pick to drop: {pick} (train NA: {a_na:.3f}/{b_na:.3f}).")

    # perform deletions in-place if requested
    dropped: List[str] = []
    if perform_delete and to_drop:
        # ensure we only drop columns that still exist
        for c in to_drop:
            if c in data.columns:
                try:
                    data.drop(columns=[c], inplace=True)
                except Exception:
                    pass
            if c in exam.columns:
                try:
                    exam.drop(columns=[c], inplace=True)
                except Exception:
                    pass
            dropped.append(c)
        if not silent:
            print(f"VIF_process: dropped {len(dropped)} columns in-place.")

    # return correlation matrix, dropped list, and confirmed pairs
    return corr_df, dropped, confirmed_pairs


### log_transform_skewed

In [7]:
def log_transform_skewed(data: pd.DataFrame, exam: pd.DataFrame, y_str: str, label_str: str, threshold: float = 2) -> bool:
    """
    Apply log transformation to all numeric columns that are present in both data and exam, if highly right-skewed.
    For the target variable y_str, always transform it in data if conditions are met, but NEVER transform it in exam.
    Returns a boolean indicating whether the target variable y was log-transformed.

    Parameters
    ----------
    data : pd.DataFrame
        Training DataFrame
    exam : pd.DataFrame
        Test DataFrame (may or may not contain y_str)
    y_str : str
        Name of the target variable
    label_str : str
        Optional label/id column in exam
    threshold : float
        Minimum skewness to trigger log transformation

    Returns
    -------
    bool
        True if y was log-transformed in data, False otherwise
    """
    # for featured columns, it should exist in both data and exam
    feature_cols = [c for c in data.select_dtypes(include=[np.number]).columns
                   if c in exam.columns and c != label_str and c != y_str]
    
    # for y_str,  only to process data[y_str] do not process exam[y_str]
    target_cols = [c for c in data.select_dtypes(include=[np.number]).columns
                  if c == y_str and c != label_str]
    
    # merge all columns to process
    cols_to_process = feature_cols + target_cols
    
    y_log_transformed = False

    for col in cols_to_process:
        train_series = data[col]
        non_null_mask = train_series.notna()
        non_null_train = train_series[non_null_mask]

        # if data is not enough/not unique, skip
        if len(non_null_train) < 10 or non_null_train.nunique() == 1 or (non_null_train <= 0).any():
            continue

        # calculate skew
        s = skew(non_null_train)
        if s <= threshold:
            continue

        # log process
        data.loc[non_null_mask, col] = np.log(non_null_train)

        # change exam as well
        if col in exam.columns and col != y_str:
            exam_series = exam[col]
            exam_non_null_mask = exam_series.notna()
            exam_positive_mask = exam_non_null_mask & (exam_series > 0)
            exam.loc[exam_positive_mask, col] = np.log(exam_series[exam_positive_mask])

        # if column is y, label true
        if col == y_str:
            y_log_transformed = True
            new_skew = skew(data.loc[non_null_mask, col])
            print(f"Target '{y_str}' log-transformed: skew {s:.4f} -> {new_skew:.4f}")

    return y_log_transformed

## data process

### substr normalize

In [8]:
def normalize_substr(data: pd.DataFrame, column: str, substr: str, mode: str = 'contain') -> None:
    """
    Clean specified column in DataFrame based on substring containment or exclusion
    
    Args:
        data: DataFrame to process
        column: Column name to process
        substr: Substring to check for
        mode: Processing mode, 'contain' or 'exclude'
    
    Returns:
        None: Modifies the input DataFrame in-place
    """
    # Check if column exists
    if column not in data.columns:
        print(f"Error: Column '{column}' not found in DataFrame")
        return
    
    # Skip if column is numeric
    if pd.api.types.is_numeric_dtype(data[column]):
        print(f"Info: Column '{column}' is numeric, skipping processing")
        return
    
    # Ensure column is string type
    if not pd.api.types.is_string_dtype(data[column]):
        data[column] = data[column].astype(str)
    
    # Process based on mode
    if mode == 'contain':
        # Keep values containing substring, set others to empty
        mask = data[column].str.contains(substr, na=False)
        data.loc[~mask, column] = pd.NA
        print(f"Processed: Kept {mask.sum()} rows containing '{substr}' in column '{column}'")
    
    elif mode == 'exclude':
        # Keep values not containing substring, set others to empty
        mask = data[column].str.contains(substr, na=False)
        data.loc[mask, column] = pd.NA
        print(f"Processed: Kept {(~mask).sum()} rows excluding '{substr}' in column '{column}'")
    
    else:
        print(f"Error: Invalid mode '{mode}'. Use 'contain' or 'exclude'")
        return

### outlier process

In [9]:
def outlier_process(
    data: pd.DataFrame,
    exam: pd.DataFrame,
    column: str,
    y_str: str = "Price",
    label_str: str = "ID",
    min_nonnull: int = 10,
    min_unique_for_continuous: int = 10,
    shapiro_sample_max: int = 5000,
    iqr_k: float = 2.0,
    sigma_k: float = 3.0,
    alpha: float = 0.05,
    verbose: bool = True
) -> None:
    """
    Apply outlier clipping to specified column in data and exam based on training set statistics.
    Skip processing for target variable columns and label columns.
    
    Key rules:
      - Calculate bounds using training set (non-null subset);
      - Skip if boolean-like or binary (0/1) columns;
      - Skip if low cardinality (<= min_unique_for_continuous), treated as discrete;
      - Otherwise: if approximately normal (Shapiro p > alpha) use sigma_k * std; else use iqr_k * IQR;
      - Use training bounds to clip exam values; no rounding (allow integer to float conversion).
    Modifies input DataFrames (data, exam) in-place, no return value.
    """
    # Skip target variable and label columns
    if column == y_str or column == label_str:
        if verbose:
            print(f"[outlier_process] Skip target/label column '{column}'")
        return
    
    # Existence check
    if column not in data.columns:
        if verbose:
            print(f"[outlier_process] column '{column}' not in data -> skip")
        return
    if column not in exam.columns:
        if verbose:
            print(f"[outlier_process] column '{column}' not in exam -> skip")
        return

    train_ser = data[column]
    test_ser = exam[column]

    # Non-null subset
    train_mask = train_ser.notna()
    train_vals = train_ser[train_mask]

    # Basic checks
    if len(train_vals) < min_nonnull:
        if verbose:
            print(f"[outlier_process] '{column}': insufficient non-null samples ({len(train_vals)}) -> skip")
        return
    if train_vals.nunique(dropna=True) == 1:
        if verbose:
            print(f"[outlier_process] '{column}': single value column -> skip")
        return

    # Check for boolean/binary (0/1) type
    unique_vals = pd.unique(train_vals.dropna())
    is_bool_like = pd.api.types.is_bool_dtype(train_ser) or set(unique_vals).issubset({0, 1})
    if is_bool_like:
        if verbose:
            print(f"[outlier_process] '{column}': boolean-like -> skip outlier processing")
        return

    # Check for low cardinality discrete columns (treated as categorical/ordinal)
    n_unique = train_vals.nunique(dropna=True)
    if n_unique <= min_unique_for_continuous:
        if verbose:
            print(f"[outlier_process] '{column}': low cardinality (n_unique={n_unique}) -> skip (treated as discrete)")
        return

    # Normality test (sample if too large)
    try:
        sample = train_vals
        if len(train_vals) > shapiro_sample_max:
            sample = train_vals.sample(n=shapiro_sample_max, random_state=111)
        _, pvalue = stats.shapiro(sample)
    except Exception:
        pvalue = 0.0  # If test fails, treat as non-normal

    # Calculate bounds (training set)
    if pvalue > alpha:
        # Approximately normal -> sigma rule
        mu = float(train_vals.mean())
        sigma = float(train_vals.std(ddof=0))
        lower = mu - sigma_k * sigma
        upper = mu + sigma_k * sigma
        method = f"{sigma_k}-sigma (normal)"
    else:
        # Non-normal -> IQR rule
        Q1 = float(train_vals.quantile(0.25))
        Q3 = float(train_vals.quantile(0.75))
        IQR = Q3 - Q1
        lower = Q1 - iqr_k * IQR
        upper = Q3 + iqr_k * IQR
        method = f"{iqr_k}-IQR (non-normal)"

    # Clip training data (preserve NaNs)
    train_before = train_vals.to_numpy(dtype=float)
    train_clipped = np.clip(train_before, lower, upper)
    # Count changed values (use isclose to tolerate float precision)
    changed_train = (~np.isclose(train_before, train_clipped, rtol=0, atol=1e-12)).sum()
    # Assign back to DataFrame
    data.loc[train_mask, column] = train_clipped

    # Clip exam data using same bounds
    test_mask = test_ser.notna()
    if test_mask.any():
        test_before = test_ser[test_mask].to_numpy(dtype=float)
        test_clipped = np.clip(test_before, lower, upper)
        changed_test = (~np.isclose(test_before, test_clipped, rtol=0, atol=1e-12)).sum()
        exam.loc[test_mask, column] = test_clipped
    else:
        changed_test = 0

    if verbose:
        print(f"[outlier_process] '{column}': method={method}; bounds=[{lower:.4f}, {upper:.4f}]")
        print(f"    training changed: {int(changed_train)} / {len(train_before)}")
        print(f"    exam changed:     {int(changed_test)} / {test_mask.sum()}")


### translate Chinese charater to number

In [10]:
def translate_chinese_to_number(chinese_str: str) -> Optional[int]:
    """
    Convert Chinese number string to integer.
    Handles: Arabic digits, 零, 一..九, 两, 十, 百, 千, 万, and combinations like "一百零三", "二万三千五百二十".
    Returns None if conversion fails (unsupported or invalid format).
    """
    if chinese_str is None:
        return None

    s = str(chinese_str).strip()
    if not s or pd.isna(s):
        return None

    # Directly return if it's purely Arabic digits
    if re.fullmatch(r'\d+', s):
        return int(s)

    chinese_to_arabic = {
        '零': 0, '一': 1, '二': 2, '三': 3, '四': 4, '五': 5,
        '六': 6, '七': 7, '八': 8, '九': 9, '十': 10, '两': 2
    }

    # Remove leading/trailing zeros, e.g., "零三" -> "三"
    s = s.strip('零')
    if not s:
        return 0

    # Handle larger units in descending order, recursively parse left and right parts
    units = [('万', 10000), ('千', 1000), ('百', 100), ('十', 10)]
    for unit_char, unit_value in units:
        if unit_char in s:
            left, right = s.split(unit_char, 1)
            # Default left to 1 if empty, e.g., "十" or "十二" means 1*10
            left_val = translate_chinese_to_number(left) if left else 1
            # Default right to 0 if empty, e.g., "二十" -> 0
            right_val = translate_chinese_to_number(right) if right else 0
            # Return None if any subpart fails, avoids arithmetic with None
            if left_val is None or right_val is None:
                return None
            return left_val * unit_value + right_val

    # Single character direct mapping
    if s in chinese_to_arabic:
        return chinese_to_arabic[s]

    # Fallback: parse character by character as decimal, e.g., "二七" -> 27
    result = 0
    for ch in s:
        if ch in chinese_to_arabic:
            result = result * 10 + chinese_to_arabic[ch]
        else:
            return None

    return result


### eraze serious missing data

In [11]:
def erase_missing_data(data: pd.DataFrame, exam: pd.DataFrame, 
                      column_ratio: float, row_ratio: float,verbose :bool=False):
    """
    Remove columns with high missing ratio from both training and exam data based on training data.
    Remove rows with high missing ratio only from training data based on training data.
    
    Args:
        data: Training DataFrame to modify in-place (both columns and rows removed)
        exam: Exam DataFrame to modify in-place (only columns removed, rows preserved)  
        column_ratio: Threshold for column missing ratio in training data (0.0 to 1.0)
        row_ratio: Threshold for row missing ratio in training data (0.0 to 1.0)
    """    
    # Step 1: Determine columns to drop based on TRAINING data only
    missing_col_ratio_train = data.isnull().mean()
    columns_to_drop = missing_col_ratio_train[missing_col_ratio_train > column_ratio].index.tolist()
    
    if columns_to_drop:
        if verbose:
            print(f"Removing {len(columns_to_drop)} columns with missing ratio > {column_ratio:.1%} in training data:")
        for col in columns_to_drop:
            missing_pct_train = missing_col_ratio_train[col] * 100
            # Also show exam data missing rate for reference
            missing_pct_exam = exam[col].isnull().mean() * 100 if col in exam.columns else 100
            print(f"  - {col}: Train({missing_pct_train:.1f}%) Exam({missing_pct_exam:.1f}%)")
        
        # Apply same column removal to both datasets
        data.drop(columns=columns_to_drop, inplace=True)
        exam.drop(columns=columns_to_drop, inplace=True)
    elif verbose:
        print(f"No columns found with missing ratio > {column_ratio:.1%} in training data")
    
    # Step 2: Remove rows with high missing ratio from TRAINING data only
    # Recalculate row missing ratio after column removal
    missing_row_ratio_train = data.isnull().mean(axis=1)
    rows_to_drop_train = missing_row_ratio_train[missing_row_ratio_train > row_ratio].index.tolist()
    
    if rows_to_drop_train:
        data.drop(index=rows_to_drop_train, inplace=True)
        if verbose:
            print(f"Removing {len(rows_to_drop_train)} rows from training data with missing ratio > {row_ratio:.1%}")
    elif verbose:
        print(f"No rows found with missing ratio > {row_ratio:.1%} in training data")
    
    # Exam data rows are NEVER removed

### filling_missing

In [12]:
def fill_missing(
    data: pd.DataFrame, 
    exam: pd.DataFrame, 
    missing_column: str,
    method: str, 
    value: Optional[Union[int, float, str]] = None,
    condition_column: Optional[str] = None,
    verbose = False
) -> None:
    """
    Fill missing values in data and exam DataFrames (modifies them in-place).
    """
    valid_methods = ['naive', 'average', 'condition_average', 'ols']
    if method not in valid_methods:
        raise ValueError(f"Method '{method}' not recognized. Valid methods: {valid_methods}")

    # Create column if it doesn't exist (will be filled later)
    if missing_column not in data.columns:
        data[missing_column] = np.nan
    if missing_column not in exam.columns:
        exam[missing_column] = np.nan

    # Normalize common missing value representations to NaN
    def _normalize_missing_strings(s: pd.Series) -> pd.Series:
        if s.dtype == object or s.dtype.name == 'string':
            s = s.replace(['', ' ', 'NA', 'N/A', 'na', 'n/a', 'nan', 'None', 'none', 'NULL', 'null'], np.nan)
        return s

    data[missing_column] = _normalize_missing_strings(data[missing_column])
    exam[missing_column] = _normalize_missing_strings(exam[missing_column])

    # Validate method-specific parameters
    if method in ['condition_average', 'ols'] and condition_column is None:
        raise ValueError(f"'{method}' method requires a condition_column parameter")
    if method in ['condition_average', 'ols'] and condition_column not in data.columns:
        raise ValueError(f"Condition column '{condition_column}' not found in data DataFrame")
    if verbose:
        print(f"Filling missing values in '{missing_column}' using method: {method}")

    data_missing_before = int(data[missing_column].isna().sum())
    exam_missing_before = int(exam[missing_column].isna().sum())

    # make sure data is float
    if pd.api.types.is_integer_dtype(data[missing_column]):
        data[missing_column] = data[missing_column].astype(float)
    if missing_column in exam.columns and pd.api.types.is_integer_dtype(exam[missing_column]):
        exam[missing_column] = exam[missing_column].astype(float)

    # Fallback value generator: try numeric mean -> mode -> 0 (or empty string for text)
    def _get_global_fallback(series: pd.Series):
        # Try numeric mean
        numeric = pd.to_numeric(series, errors='coerce')
        if numeric.notna().sum() > 0:
            return float(numeric.mean())
        # Try mode (original type)
        try:
            mode = series.mode(dropna=True)
            if len(mode) > 0:
                return mode.iloc[0]
        except Exception:
            pass
        # Final fallback to 0 (acceptable for numeric features)
        return 0

    if method == 'naive':
        if value is None:
            raise ValueError("'naive' method requires a value parameter")
        fill_val = value
        # Explicit assignment
        data.loc[:, missing_column] = data[missing_column].fillna(fill_val)
        exam.loc[:, missing_column] = exam[missing_column].fillna(fill_val)

    elif method == 'average':
        # Calculate mean based on data (prioritize numeric mean)
        numeric = pd.to_numeric(data[missing_column], errors='coerce')
        if numeric.notna().sum() > 0:
            mean_value = float(numeric.mean())
            fill_val = mean_value
            data.loc[:, missing_column] = data[missing_column].fillna(fill_val)
            exam.loc[:, missing_column] = exam[missing_column].fillna(fill_val)
            print(f"  Used numeric mean value from data: {fill_val:.6f}")
        else:
            # Fallback to mode
            mode = data[missing_column].mode(dropna=True)
            if len(mode) > 0:
                fill_val = mode.iloc[0]
                data.loc[:, missing_column] = data[missing_column].fillna(fill_val)
                exam.loc[:, missing_column] = exam[missing_column].fillna(fill_val)
                print(f"  No numeric values: used mode value from data: {fill_val}")
            else:
                # Final fallback
                fill_val = _get_global_fallback(data[missing_column])
                data.loc[:, missing_column] = data[missing_column].fillna(fill_val)
                exam.loc[:, missing_column] = exam[missing_column].fillna(fill_val)
                print(f"  No numeric/mode values: used fallback value: {fill_val}")

    elif method == 'condition_average':
        # Calculate group-wise fill values in data (prioritize numeric mean)
        group_means = {}
        for name, grp in data.groupby(condition_column):
            numeric_grp = pd.to_numeric(grp[missing_column], errors='coerce')
            if numeric_grp.notna().sum() > 0:
                group_means[name] = float(numeric_grp.mean())
            else:
                # Fallback to group mode
                mode = grp[missing_column].mode(dropna=True)
                if len(mode) > 0:
                    group_means[name] = mode.iloc[0]
                else:
                    # Mark as None, will use global fallback later
                    group_means[name] = None

        # Global fallback (for groups with None or exam conditions not in data)
        global_fallback = _get_global_fallback(data[missing_column])

        # Map back to data and exam: prioritize group_means, otherwise use global_fallback
        # Fill missing values in data based on groups
        data_missing_mask = data[missing_column].isna()
        if data_missing_mask.any():
            # Map will create NaN for groups with None or non-existent conditions
            mapped = data.loc[data_missing_mask, condition_column].map(group_means)
            # Replace None in group_means with global_fallback
            mapped = mapped.fillna(global_fallback)
            data.loc[data_missing_mask, missing_column] = mapped.values

        # Fill missing values in exam based on condition (exam's condition_column might not exist or be NaN)
        if condition_column not in exam.columns:
            exam[condition_column] = np.nan
        exam_missing_mask = exam[missing_column].isna()
        if exam_missing_mask.any():
            mapped_exam = exam.loc[exam_missing_mask, condition_column].map(group_means)
            mapped_exam = mapped_exam.fillna(global_fallback)
            exam.loc[exam_missing_mask, missing_column] = mapped_exam.values
        if verbose:
            print(f"  Calculated conditional means for {len(group_means)} groups; used global fallback {global_fallback}")

    elif method == 'ols':
        # Check if both condition and missing columns are numeric
        if not (pd.api.types.is_numeric_dtype(data[condition_column]) and 
                pd.api.types.is_numeric_dtype(data[missing_column])):
            print("Warning: OLS method requires both condition and missing columns to be numeric.")
            print("  Falling back to condition_average method.")
            # Fall back to condition_average method
            fill_missing(data, exam, missing_column, 'condition_average', value, condition_column)
            return
        
        # Prepare training data: rows where both columns are not missing
        train_mask = data[missing_column].notna() & data[condition_column].notna()
        if train_mask.sum() < 2:  # Need at least 2 points to fit a line
            print("Warning: Insufficient data for OLS regression (need at least 2 non-missing pairs).")
            print("  Falling back to condition_average method.")
            fill_missing(data, exam, missing_column, 'condition_average', value, condition_column)
            return
        
        # Extract training data
        X_train = data.loc[train_mask, condition_column].values.reshape(-1, 1)
        y_train = data.loc[train_mask, missing_column].values
        
        # Train OLS model
        try:
            model = LinearRegression()
            model.fit(X_train, y_train)
            
            # Fill missing values in training set
            data_missing_mask = data[missing_column].isna() & data[condition_column].notna()
            if data_missing_mask.any():
                X_pred = data.loc[data_missing_mask, condition_column].values.reshape(-1, 1)
                data.loc[data_missing_mask, missing_column] = model.predict(X_pred)
            
            # Fill missing values in exam set
            exam_missing_mask = exam[missing_column].isna() & exam[condition_column].notna()
            if exam_missing_mask.any():
                X_exam_pred = exam.loc[exam_missing_mask, condition_column].values.reshape(-1, 1)
                exam.loc[exam_missing_mask, missing_column] = model.predict(X_exam_pred)
            
            print(f"  Used OLS regression: y = {model.coef_[0]:.4f} * x + {model.intercept_:.4f}")
            print(f"  R² score on training data: {model.score(X_train, y_train):.4f}")
            
        except Exception as e:
            print(f"Warning: OLS regression failed: {e}")
            print("  Falling back to condition_average method.")
            fill_missing(data, exam, missing_column, 'condition_average', value, condition_column)
            return

    # Final safety: if any NaN remains (edge cases), use global fallback to ensure no NaN remains
    final_fill = _get_global_fallback(data[missing_column])
    data.loc[:, missing_column] = data[missing_column].fillna(final_fill)
    exam.loc[:, missing_column] = exam[missing_column].fillna(final_fill)

    data_missing_after = int(data[missing_column].isna().sum())
    exam_missing_after = int(exam[missing_column].isna().sum())

    # Verify no missing values remain
    assert data_missing_after == 0, f"Data still contains {data_missing_after} missing values in {missing_column}"
    assert exam_missing_after == 0, f"Exam still contains {exam_missing_after} missing values in {missing_column}"
    if verbose:
        print(f"Fill results:")
        print(f"  - Data: {data_missing_before} → {data_missing_after} missing values")
        print(f"  - Exam: {exam_missing_before} → {exam_missing_after} missing values")
        print(f"  - Successfully eliminated all missing values from '{missing_column}'")

### first numeric clean

numeric_extract_clean function apply to:
房屋总数	楼栋总数    绿 化 率   建筑面积	   套内面积
These data share one feature: number+unit

In [13]:
def first_numeric_clean(df: pd.DataFrame, column: str):
    """
    General function to extract single numeric values from text columns.
    Handles patterns like: '19栋', '30%', '52.3㎡', '1317户'
    Modify the input DataFrame in-place.
    
    Args:
        df: Input pandas DataFrame
        column: Name of the column to process
    """
    if column not in df.columns:
        print(f"Error: Column '{column}' not found in DataFrame")
        return
    
    numeric_values = []
    
    for value in df[column]:
        if pd.isna(value):
            numeric_values.append(None)
            continue
            
        value_str = str(value).strip()
        
        # Pattern for extracting numbers (including decimals)
        match = re.search(r'(\d+\.?\d*)', value_str)
        
        if match:
            try:
                numeric_value = float(match.group(1))
                numeric_values.append(numeric_value)
            except (ValueError, TypeError):
                numeric_values.append(None)
        else:
            numeric_values.append(None)
    
    # Replace the original column
    df[column] = numeric_values

    print(f'first_numeric_clean extracting numbers in {column}, finished.')

### numeric_region_clean

Some data take this form:
1.23 m/s
24-25
use region extrat to solve this

In [14]:
def numeric_range_clean(df: pd.DataFrame, column: str, method: str) -> None:
    """
    Extract numeric values from range-type text columns.
    For ranges like '2.61-2.63', extract the middle value or split into two columns.
    Modify the input DataFrame in-place.
    
    Args:
        df: Input pandas DataFrame
        column: Name of the column to process
        method: 'medium' to use middle value, 'split' to create two columns
    """
    if column not in df.columns:
        print(f"Error: Column '{column}' not found in DataFrame")
        return
    
    # Fetch low and high
    all_lows = []
    all_highs = []
    for value in df[column]:
        if pd.isna(value):
            all_lows.append(None)
            all_highs.append(None)
            continue
            
        value_str = str(value).strip()
        
        range_match = re.search(r'(\d+\.?\d*)\s*-\s*(\d+\.?\d*)', value_str)
        single_match = re.search(r'(\d+\.?\d*)', value_str)
        
        if range_match:
            try:
                low = float(range_match.group(1))
                high = float(range_match.group(2))
                all_lows.append(low)
                all_highs.append(high)
            except (ValueError, TypeError):
                all_lows.append(None)
                all_highs.append(None)
        elif single_match:
            try:
                single = float(single_match.group(1))
                all_lows.append(single)  # 单个值视为low=high
                all_highs.append(single)
            except (ValueError, TypeError):
                all_lows.append(None)
                all_highs.append(None)
        else:
            all_lows.append(None)
            all_highs.append(None)
    
    if method == 'medium':
        # 基于统一提取的low和high计算中间值
        middle_values = []
        for low, high in zip(all_lows, all_highs):
            if low is not None and high is not None:
                middle_values.append((low + high) / 2)
            else:
                middle_values.append(None)
        
        df[column] = middle_values
        
    elif method == 'split':
        # 直接使用统一提取的low和high
        df[f'{column}_low'] = all_lows
        df[f'{column}_high'] = all_highs
        df.drop(columns=[column], inplace=True)
    
    else:
        print(f"Error: Unknown method '{method}'. Use 'medium' or 'split'.")
        return

### extract_dummy_substr_split

In [15]:
def extract_dummy_substr_split(data: pd.DataFrame, exam: pd.DataFrame, column_name: str, 
                        separator: str, discard_str: Optional[str] = None, silent: bool = True) -> None:
    """
    Extract dummy variables from a column containing multiple substrings separated by a delimiter.
    Creates dummy variables for each unique substring found in the data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Training dataset (modified in-place)
    exam : pd.DataFrame
        Exam dataset (modified in-place)
    column_name : str
        Column name to extract dummy variables from
    separator : str, default '/'
        Delimiter used to separate substrings in the column
    discard_str : str, optional
        String to discard from substrings (e.g., '结合' in '塔板结合')
    silent : bool, default True
        If True, suppress print outputs
        
    Returns:
    --------
    None
        Both DataFrames are modified directly
    """
    
    # check whether column exit in both data and exam
    if (column_name not in data.columns) or (column_name not in exam.columns):
        if not silent:
            print(f"Error: Column '{column_name}' not found in data or exam set")
        return
    
    # Only extract substring in data not exam
    all_substrings = set()
    for value in data[column_name]:
        if pd.isna(value):
            continue
        substrings = _extract_substrings(value, separator, discard_str)
        all_substrings.update(substrings)
    
    all_substrings = sorted(list(all_substrings))
    
    # if no valid data, quit
    if not all_substrings:
        if not silent:
            print(f"No valid substrings found in column '{column_name}'")
        return
    
    if not silent:
        print(f"Found {len(all_substrings)} unique substrings in '{column_name}': {all_substrings}")
    
    # create dummy variable for data and exam
    for substring in all_substrings:
        dummy_col_name = f"{column_name}_{substring}"
        data[dummy_col_name] = data[column_name].apply(
            lambda x: 1 if _contains_substring(x, substring, separator, discard_str) else 0
        )
        exam[dummy_col_name] = exam[column_name].apply(
            lambda x: 1 if _contains_substring(x, substring, separator, discard_str) else 0
        )
        
    # delete original column
    data.drop(columns=[column_name], inplace=True)
    exam.drop(columns=[column_name], inplace=True)
    
    if not silent:
        print(f"Created {len(all_substrings)} dummy variables for '{column_name}'")


def _extract_substrings(value: Optional[Any], separator: str, discard_str: Optional[str] = None) -> List[str]:
    """
    Extract and clean substrings from a value using the separator.
    
    Parameters:
    -----------
    value : str
        Original string value
    separator : str
        Delimiter used to separate substrings
    discard_str : str, optional
        String to discard from substrings
        
    Returns:
    --------
    list
        List of cleaned substrings
    """
    if pd.isna(value):
        return []
    
    # Ensure we work with a str (handles numbers, etc.)
    value_str = str(value)
    
    # Split by separator
    substrings = value_str.split(separator)
    
    # Clean each substring
    cleaned_substrings: List[str] = []
    for substr in substrings:
        # Remove whitespace
        substr = substr.strip()
        
        # Discard specified string if present
        if discard_str and discard_str in substr:
            substr = substr.replace(discard_str, '')
            substr = substr.strip()
        
        # Skip empty strings
        if substr:
            cleaned_substrings.append(substr)
    
    return cleaned_substrings

def _contains_substring(value: Optional[Any], substring: str, separator: str, discard_str: Optional[str] = None) -> bool:
    """
    Check if a value contains a specific substring.
    
    Parameters:
    -----------
    value : str
        Original string value
    substring : str
        Substring to check for
    separator : str
        Delimiter used in the original value
    discard_str : str, optional
        String to discard from substrings
        
    Returns:
    --------
    bool
        True if the substring is found in the value
    """
    if pd.isna(value):
        return False
    
    extracted = _extract_substrings(value, separator, discard_str)
    return substring in extracted


### identical column process

**example:**
板块<->板块_comm
环线<->环线位置 
区县<->区域
lon<->coord_x
lat<->coord_y

These column are quite similar

In [16]:
def identical_column_test(data: pd.DataFrame, col1: str, col2: str, show_examples: int = 0, ratio: float = 0.05):
    """
    Test if two columns are essentially identical.
    
    Args:
        data: Input DataFrame
        col1: First column name
        col2: Second column name
        show_examples: Number of different examples to show (default 0 = don't show, -1 = show all)
        ratio: Threshold for considering columns identical (default 0.05 = 5%)
    
    Returns:
        Tuple of (is_identical, actual_difference_ratio)
    """
    if col1 not in data.columns or col2 not in data.columns:
        print(f"Error: One or both columns not found: {col1}, {col2}")
        return False, 1.0
    
    # Get rows where both columns have values
    both_notna = data[[col1, col2]].notna().all(axis=1)
    
    if both_notna.sum() == 0:
        print(f"No rows with both {col1} and {col2} having values")
        return False, 1.0
    
    comparison_data = data[both_notna]
    
    # Check if columns are numeric
    col1_numeric = pd.api.types.is_numeric_dtype(comparison_data[col1])
    col2_numeric = pd.api.types.is_numeric_dtype(comparison_data[col2])
    
    if col1_numeric and col2_numeric:
        # For numeric columns, use absolute difference < 0.001
        differences = np.abs(comparison_data[col1] - comparison_data[col2]) > 0.001
        diff_values = comparison_data[differences][[col1, col2]].copy()
        diff_values['difference'] = np.abs(diff_values[col1] - diff_values[col2])
        
        # Calculate difference statistics
        max_diff = diff_values['difference'].max() if len(diff_values) > 0 else 0
        mean_diff = diff_values['difference'].mean() if len(diff_values) > 0 else 0
        std_diff = diff_values['difference'].std() if len(diff_values) > 0 else 0
        
    else:
        # For non-numeric, use exact equality
        differences = comparison_data[col1] != comparison_data[col2]
        diff_values = comparison_data[differences][[col1, col2]].copy()
        diff_values['difference'] = "string_diff"
        max_diff = mean_diff = std_diff = None
    
    diff_ratio = differences.sum() / len(comparison_data)
    is_identical = diff_ratio <= ratio
    
    print(f"Column comparison: {col1} vs {col2}")
    print(f"  Rows with both values: {len(comparison_data)}")
    print(f"  Different rows: {differences.sum()}")
    print(f"  Difference ratio: {diff_ratio:.4f} ({diff_ratio*100:.2f}%)")
    print(f"  Identical (≤{ratio*100:.1f}%): {is_identical}")
    
    # Show numeric difference statistics if applicable
    if col1_numeric and col2_numeric and differences.sum() > 0:
        print(f"  Difference statistics:")
        print(f"    Max difference: {max_diff:.6f}")
        print(f"    Mean difference: {mean_diff:.6f}")
        print(f"    Std difference: {std_diff:.6f}")
    
    # Show examples of differences if requested
    if show_examples != 0 and differences.sum() > 0:
        num_examples = differences.sum() if show_examples == -1 else min(show_examples, differences.sum())
        print(f"\n  Examples of differences (showing {num_examples}):")
        
        examples_to_show = diff_values.head(num_examples)
        for idx, row in examples_to_show.iterrows():
            if col1_numeric and col2_numeric:
                print(f"    Row {idx}: {col1}={row[col1]}, {col2}={row[col2]}, diff={row['difference']:.6f}")
            else:
                print(f"    Row {idx}: {col1}='{row[col1]}', {col2}='{row[col2]}'")
    
    print("")
    
    return is_identical, diff_ratio


def merge_identical_column(
    data: pd.DataFrame,
    col1: str,
    col2: str,
    method: str = 'average',
    numeric_tolerance: float = 0.001,
    verbose = False
):
    """
    Merge two columns and delete the second one.
    - Fill missing values in col1 from col2.
    - For conflicting values:
        * if method='average':
            - if both can be parsed as numeric -> if difference > numeric_tolerance => average (preserve int if both ints)
            - otherwise -> choose value from the column with lower missing rate (if equal, keep col1)
        * if method='select':
            - always choose value from the column with lower missing rate (if equal, keep col1)
    The input DataFrame is modified in-place and col2 is dropped.

    Args:
        data: DataFrame to modify (in-place).
        col1: primary column to keep (name preserved).
        col2: secondary column to merge and delete.
        method: conflict resolution method - 'average' or 'select'
        numeric_tolerance: threshold to decide if two numeric values differ (only for method='average').
        verbose :print more information or not
    """
    if col1 not in data.columns or col2 not in data.columns:
        print(f"Error: One or both columns not found: {col1}, {col2}")
        return
    
    if method not in ['average', 'select']:
        print(f"Error: method must be 'average' or 'select', got '{method}'")
        return

    col1_missing_rate = data[col1].isna().mean()
    col2_missing_rate = data[col2].isna().mean()
    if verbose:
        print(f"Data quality: {col1} missing rate: {col1_missing_rate:.2%}, {col2} missing rate: {col2_missing_rate:.2%}")
        print(f"Using conflict resolution method: {method}")

    n_rows = len(data)

    # 1) Fill missing values in col1 from col2
    mask_fill = data[col1].isna() & data[col2].notna()
    n_filled = int(mask_fill.sum())
    if n_filled > 0:
        data.loc[mask_fill, col1] = data.loc[mask_fill, col2]
        print(f"Filled {n_filled} missing values in {col1} from {col2}")

    # 2) Find rows where both are present
    both_present = data[col1].notna() & data[col2].notna()
    if not both_present.any():
        # nothing more to do
        data.drop(columns=[col2], inplace=True)
        if verbose:
            print(f"Deleted column: {col2}")
        final_missing = data[col1].isna().sum()
        if verbose:
            print(f"Final missing values in {col1}: {final_missing} ({final_missing / n_rows:.2%})")
        return

    col1_vals_raw = data.loc[both_present, col1]
    col2_vals_raw = data.loc[both_present, col2]

    if method == 'average':
        col1_num = pd.to_numeric(col1_vals_raw, errors='coerce')
        col2_num = pd.to_numeric(col2_vals_raw, errors='coerce')

        # numeric candidates
        numeric_both = col1_num.notna() & col2_num.notna()

        # 2a) Handle numeric conflicts
        conflict_idx: pd.Index = pd.Index([])
        if numeric_both.any():
            diff = (col1_num[numeric_both] - col2_num[numeric_both]).abs()
            numeric_conflict_mask = diff > numeric_tolerance
            if numeric_conflict_mask.any():
                conflict_idx = numeric_conflict_mask[numeric_conflict_mask].index
                avg_values = (col1_num.loc[conflict_idx] + col2_num.loc[conflict_idx]) / 2.0

                # preserve integer type if original columns are int dtype
                if pd.api.types.is_integer_dtype(data[col1]) and pd.api.types.is_integer_dtype(data[col2]):
                    avg_values = avg_values.round().astype(int)

                data.loc[conflict_idx, col1] = avg_values
                print(f"Averaged {len(conflict_idx)} conflicting numeric values between {col1} and {col2}")

        # 2b) Handle remaining conflicts (non-numeric)
        both_present_idx = pd.Index(data.index[both_present])
        neq_mask = data.loc[both_present_idx, col1] != data.loc[both_present_idx, col2]

        remaining_conflict_idx = pd.Index([idx for idx, neq in neq_mask.items() if neq and idx not in conflict_idx])

        if len(remaining_conflict_idx) > 0:
            remaining_count = len(remaining_conflict_idx)
            # choose value from column with lower missing rate, if equal keep col1
            if col2_missing_rate < col1_missing_rate:
                data.loc[remaining_conflict_idx, col1] = data.loc[remaining_conflict_idx, col2]
                print(f"For {remaining_count} non-numeric conflicts, chose values from {col2} (lower missing rate).")
            else:
                print(f"For {remaining_count} non-numeric conflicts, kept existing values from {col1} (equal/better missing rate).")
    
    elif method == 'select':
        # For 'select' method, handle all conflicts by choosing the column with lower missing rate
        both_present_idx = pd.Index(data.index[both_present])
        neq_mask = data.loc[both_present_idx, col1] != data.loc[both_present_idx, col2]
        
        conflict_idx = neq_mask[neq_mask].index
        
        if len(conflict_idx) > 0:
            conflict_count = len(conflict_idx)
            # choose value from column with lower missing rate, if equal keep col1
            if col2_missing_rate < col1_missing_rate:
                data.loc[conflict_idx, col1] = data.loc[conflict_idx, col2]
                print(f"For {conflict_count} conflicts, chose values from {col2} (lower missing rate).")
            else:
                print(f"For {conflict_count} conflicts, kept existing values from {col1} (equal/better missing rate).")

    # 3) Drop col2
    data.drop(columns=[col2], inplace=True)
    print(f"Deleted column: {col2}")
    
    final_missing = data[col1].isna().sum()
    print(f"Final missing values in {col1}: {final_missing} ({final_missing / n_rows:.2%})")

### to_dummy

In [17]:
def _replace_dataframe_contents_inplace(target: pd.DataFrame, newframe: pd.DataFrame) -> None:
    """
    Replace contents of `target` with `newframe` while keeping the same Python object.
    Primary method: swap internal manager (fast, avoids fragmentation).
    Fallbacks try to minimize per-column inserts.
    """
    try:
        target._mgr = newframe._mgr
        target.attrs = newframe.attrs
    except Exception:
        try:
            newframe = newframe.reindex(index=target.index)
            for c in [c for c in target.columns if c not in newframe.columns]:
                target.drop(columns=[c], inplace=True)
            target.loc[:, newframe.columns] = newframe
            for c in [c for c in newframe.columns if c not in target.columns]:
                if c not in target.columns:
                    target[c] = newframe[c].values
        except Exception:
            target.drop(columns=list(target.columns), inplace=True)
            for col in newframe.columns:
                target[col] = newframe[col].values
            target.index = newframe.index

def to_dummy_multiple(data: pd.DataFrame, exam: pd.DataFrame, columns: List[str],
                      handle_na: str = 'warn', drop_first: bool = True,silent = False) -> None:
    """
    Convert multiple categorical variables to dummy variables in-place for both `data` and `exam`.
    Minimal-fix behavior: dummy columns are generated *only* from training set categories.
    Any categories present only in `exam` will NOT create new dummy columns and will be encoded as all-zeros.
    """
    if not isinstance(columns, (list, tuple)):
        raise TypeError("columns must be a list or tuple of column names")

    processed_cols: List[str] = []
    train_dummies_list: List[pd.DataFrame] = []
    exam_dummies_list: List[pd.DataFrame] = []
    total_created = 0

    for column in columns:
        if column not in data.columns:
            print(f"Warning: Column '{column}' not found in training data, skipping")
            continue

        # copy series so we don't mutate original mid-loop
        train_s = data[column].copy()
        exam_s = exam[column].copy() if column in exam.columns else pd.Series([pd.NA] * len(exam), index=exam.index)

        # handle NA option
        if handle_na == 'category':
            train_s = train_s.fillna('_MISSING_')
            exam_s = exam_s.fillna('_MISSING_')
        elif handle_na == 'warn':
            train_na = int(train_s.isna().sum())
            exam_na = int(exam_s.isna().sum())
            if train_na > 0 or exam_na > 0:
                print(f"\aWarning: Column '{column}' has {train_na} missing in training, {exam_na} in exam")
        else:
            raise ValueError("handle_na must be either 'warn' or 'category'")

        # collect categories (as strings for stable colnames)
        if handle_na == 'category':
            cats_train = pd.Index(train_s.unique())
        else:
            cats_train = pd.Index(train_s.dropna().unique())

        # *** MINIMAL FIX: only use training categories to decide dummy columns ***
        # If training set has no categories at all, skip to avoid using exam-only categories.
        if len(cats_train) == 0:
            print(f"Warning: Column '{column}' has no categories in training set; skipping to avoid using exam-only categories")
            continue

        # deterministic order: use train categories only
        all_cats = list(map(str, cats_train.tolist()))
        dummy_col_names = [f"{column}_{cat}" for cat in all_cats]

        # build dummies: create columns only for training categories; exam unseen categories will be all-zeros
        train_dummies = pd.get_dummies(train_s.astype(str), prefix=column).reindex(columns=dummy_col_names, fill_value=0).astype(int)
        exam_dummies = pd.get_dummies(exam_s.astype(str), prefix=column).reindex(columns=dummy_col_names, fill_value=0).astype(int)

        # determine first category to drop if requested (based on training categories only)
        first_to_drop = None
        if drop_first and len(all_cats) > 0:
            if handle_na == 'category' and '_MISSING_' in all_cats:
                non_missing = [c for c in all_cats if c != '_MISSING_']
                if non_missing:
                    first_to_drop = non_missing[0]
            else:
                first_to_drop = all_cats[0]

        if first_to_drop is not None:
            drop_col = f"{column}_{first_to_drop}"
            if drop_col in train_dummies.columns:
                train_dummies = train_dummies.drop(columns=[drop_col])
            if drop_col in exam_dummies.columns:
                exam_dummies = exam_dummies.drop(columns=[drop_col])

        # store results for later bulk concat
        processed_cols.append(column)
        train_dummies_list.append(train_dummies)
        exam_dummies_list.append(exam_dummies)
        total_created += train_dummies.shape[1]

    # if nothing processed, exit early
    if len(processed_cols) == 0:
        return

    # build new training DataFrame: keep original columns except processed ones, then append all dummies
    other_train_cols = [c for c in data.columns if c not in processed_cols]
    left_train = data[other_train_cols].copy()
    all_train_dummies = pd.concat(train_dummies_list, axis=1) if train_dummies_list else pd.DataFrame(index=data.index)
    all_train_dummies = all_train_dummies.reindex(index=data.index).fillna(0).astype(int)
    new_train = pd.concat([left_train, all_train_dummies], axis=1)

    # build new exam DataFrame similarly (exam will have the same dummy columns; unseen exam categories will be zeros)
    other_exam_cols = [c for c in exam.columns if c not in processed_cols]
    left_exam = exam[other_exam_cols].copy()
    all_exam_dummies = pd.concat(exam_dummies_list, axis=1) if exam_dummies_list else pd.DataFrame(index=exam.index)
    all_exam_dummies = all_exam_dummies.reindex(index=exam.index).fillna(0).astype(int)
    new_exam = pd.concat([left_exam, all_exam_dummies], axis=1)

    # replace contents in-place (keep data and exam the same Python objects)
    _replace_dataframe_contents_inplace(data, new_train)
    _replace_dataframe_contents_inplace(exam, new_exam)
    if not silent:
        print(f"Created {total_created} dummy variables for columns: {processed_cols}")


### area_clean

As "identical_column_process" menthoned, data 区域 区县 板块 板块_comm 环线 环线位置 store almost identical data with a little contradiction.  
Therefore, this part use KNN to solve this contradiction.  
data related to area:
- 城市
- 板块<->板块_comm
- 环线<->环线位置 
- 区县<->区域
- lon<->coord_x
- lat<->coord_y
note: this issue mainly appear in price, less in rent.

In [18]:
def probabilistic_hierarchy_validation(
    data: pd.DataFrame, 
    parent_col: str, 
    child_col: str, 
    threshold: float = 0.9, 
    print_detail: bool = False
) -> float:
    """
    Validate if there is a hierarchical relationship between two columns
    
    Parameters:
    -----------
    data : pd.DataFrame
        Input dataframe
    parent_col : str
        Parent column name (higher level)
    child_col : str 
        Child column name (lower level)
    threshold : float, default=0.9
        Threshold for single child mapping ratio to most frequent parent
    print_detail : bool, default=False
        Whether to output detailed results to txt file
        
    Returns:
    --------
    float
        Hierarchical relationship strength (0 to 1)
        -1.0: Parent column not found
        -2.0: Child column not found
        -3.0: Insufficient data with both columns present
    """
    
    # Check if columns exist
    if parent_col not in data.columns:
        return -1.0
    if child_col not in data.columns:
        return -2.0
    
    # Filter rows where both parent and child have values
    valid_data = data[[parent_col, child_col]].dropna()
    if len(valid_data) < 100:
        return -3.0
    
    # Get unique values
    unique_children = valid_data[child_col].unique()
    unique_parents = valid_data[parent_col].unique()
    
    # Calculate mapping relationships
    child_parent_mapping = {}
    hierarchy_info = {}
    
    for child in unique_children:
        child_data = valid_data[valid_data[child_col] == child]
        parent_counts = child_data[parent_col].value_counts()
        
        total_count = parent_counts.sum()
        max_parent = parent_counts.index[0]
        max_count = parent_counts.iloc[0]
        mapping_strength = max_count / total_count
        
        child_parent_mapping[child] = max_parent
        hierarchy_info[child] = {
            'dominant_parent': max_parent,
            'max_count': max_count,
            'total_count': total_count,
            'mapping_strength': mapping_strength
        }
    
    # Calculate hierarchical relationship strength
    strong_mappings = [child for child, info in hierarchy_info.items() 
                      if info['mapping_strength'] >= threshold]
    
    hierarchy_strength = len(strong_mappings) / len(unique_children) if len(unique_children) > 0 else 0.0
    result = hierarchy_strength 
    
    # Generate detailed report if requested
    if print_detail:
        filename = f"{parent_col}_{child_col}_geolocation.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(f"Child unique values count: {len(unique_children)}\n")
            f.write(f"Parent unique values count: {len(unique_parents)}\n")
            f.write(f"Threshold: {threshold}\n")
            f.write(f"Hierarchical relationship strength: {hierarchy_strength:.3f}\n")
            f.write("Hierarchical relationships:\n")
            f.write("Dominant mapping, Mapping count, Child occurrence count, Mapping strength\n")
            
            for child, info in hierarchy_info.items():
                f.write(f"{child} -> {info['dominant_parent']}, {info['max_count']}, {info['total_count']}, {info['mapping_strength']:.2f}\n")
    
    return result

In [19]:
# try to use ANOVA, to check whether plate is significant or not.
# If this is not significant, we can choose to delete plate direcitly.
# Although data is significant, I still delete it, because there is no enough computing power.
def two_way_ANOVA(data, county_col, community_col, area_col, price_col,
                  min_samples=100, alpha=0.05):
    """
    Two-way ANOVA: 控制县 (county)，检验板块 (plate) 显著性
    - 小样本 county/plate 自动归为 "Other"
    - 回归单位面积价格 (price/area)
    """
    import pandas as pd
    import numpy as np
    import statsmodels.formula.api as smf
    import statsmodels.api as sm

    # 复制数据并计算 unit_price
    df = data[[county_col, community_col, area_col, price_col]].copy()
    df = df.dropna(subset=[county_col, community_col, area_col, price_col])
    df = df[df[area_col] > 0]
    df['unit_price'] = df[price_col] / df[area_col]

    # 处理小样本 county
    county_counts = df[county_col].value_counts()
    small_counties = county_counts[county_counts < min_samples].index
    df[county_col] = df[county_col].replace(small_counties, 'Other')

    # 处理小样本 community
    community_counts = df.groupby([county_col, community_col]).size()
    small_communities = community_counts[community_counts < min_samples].index
    for cty, comm in small_communities:
        mask = (df[county_col] == cty) & (df[community_col] == comm)
        df.loc[mask, community_col] = 'Other'

    # 重命名列以匹配 formula
    df = df.rename(columns={county_col: 'county', community_col: 'community'})

    print(f"处理后：county={df['county'].nunique()} 组，community={df['community'].nunique()} 组")

    # two-way ANOVA
    formula = 'unit_price ~ C(county) + C(community)'
    try:
        model = smf.ols(formula, data=df).fit()
        anova_res = sm.stats.anova_lm(model, typ=2)
        print("\nTwo-way ANOVA 结果：")
        print(anova_res.to_string())

        pval = anova_res.loc['C(community)', 'PR(>F)']
        if pval < alpha:
            print(f"\nCommunity 显著 (p={pval:.4g}) → 保留 community")
        else:
            print(f"\nCommunity 不显著 (p={pval:.4g}) → 可以丢掉 community")
    except Exception as e:
        print("ANOVA 执行失败:", e)
        print("可能是特征矩阵过大或奇异，可进一步合并小样本社区/县或使用 mean encoding")

In [20]:
def knn_fill_categorical(data, exam, columns_to_fill=['county', 'plate', 'ring'],verbose:bool=False):
    """
    Use lontitude and latitude to fill empty in county, plate, ring
    Changes will directly apply to input argument.
    """
    
    # Process data and exam
    for dataset_name, dataset in [('data', data), ('exam', exam)]:
        
        for col in columns_to_fill:
            if col not in dataset.columns:
                continue
                
            # Transfer data to string
            dataset[col] = dataset[col].astype(str)
            
            # label places to fill
            need_fill_mask = (dataset[col] == '0') | (dataset[col] == '-1') | (dataset[col] == 'nan') | (dataset[col].isna())
            known_mask = ~need_fill_mask
            
            if need_fill_mask.sum() == 0:
                if verbose:
                    print(f"  {col}: Nothing to fill in")
                continue
            
            try:
                # Use longtitude and latitude to predict
                X_train = dataset.loc[known_mask, ['lon', 'lat']].values
                y_train = dataset.loc[known_mask, col].values
                X_pred = dataset.loc[need_fill_mask, ['lon', 'lat']].values
                
                # Create KNN filter
                k = min(5, len(X_train))
                knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
                knn.fit(X_train, y_train)
                
                # predict missing data
                predictions = knn.predict(X_pred)
                dataset.loc[need_fill_mask, col] = predictions
                                
            except Exception as e:
                if verbose:
                    print(f" using mode to fill missing ,because KNN faild{e}")
                mode_val = dataset.loc[known_mask, col].mode()
                if len(mode_val) > 0:
                    dataset.loc[need_fill_mask, col] = mode_val.iloc[0]

### floor_clean

In [21]:
def floor_clean(df: pd.DataFrame, floor_column: str, verbose: bool = False):
    """
    Floor data cleaning function (with debug option).
    - Supports multiple Chinese floor description formats.
    - Fixes regex capture group warnings.
    - Optionally outputs unprocessed entries when debug=True.

    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to clean.
    floor_column : str
        The name of the column containing floor information.
    debug : bool, default=False
        If True, prints entries that could not be parsed.
    """
    # Check if target column exists
    if floor_column not in df.columns:
        print(f"Error: Column '{floor_column}' not found in DataFrame")
        return

    # Convert to string for consistent processing
    floor_series = df[floor_column].astype(str)

    # Initialize output columns with proper nullable dtypes
    df['current_floor'] = pd.Series([pd.NA] * len(df), index=df.index, dtype='object')
    df['total_floor'] = pd.Series([pd.NA] * len(df), index=df.index, dtype='Int32')

    # pattern1: "中楼层(共23层)"
    pattern1_mask = floor_series.str.contains(
        r'(?:地下室|底层|低楼层|中楼层|高楼层|顶层)\s*\(\s*共\s*\d+\s*层\s*\)', na=False
    )
    if pattern1_mask.any():
        df.loc[pattern1_mask, 'current_floor'] = floor_series[pattern1_mask].str.extract(
            r'(地下室|底层|低楼层|中楼层|高楼层|顶层)'
        )[0]
        df.loc[pattern1_mask, 'total_floor'] = floor_series[pattern1_mask].str.extract(
            r'共\s*(\d+)\s*层'
        )[0].astype('Int32')

    # pattern2: "低楼层/18层"
    pattern2_mask = ~pattern1_mask & floor_series.str.contains(
        r'(?:地下室|底层|低楼层|中楼层|高楼层|顶层)\s*/\s*\d+层', na=False
    )
    if pattern2_mask.any():
        df.loc[pattern2_mask, 'current_floor'] = floor_series[pattern2_mask].str.extract(
            r'(地下室|底层|低楼层|中楼层|高楼层|顶层)'
        )[0]
        df.loc[pattern2_mask, 'total_floor'] = floor_series[pattern2_mask].str.extract(
            r'/\s*(\d+)层'
        )[0].astype('Int32')

    # pattern3: "14/26层"
    pattern3_mask = ~pattern1_mask & ~pattern2_mask & floor_series.str.contains(
        r'\d+\s*/\s*\d+\s*层', na=False
    )
    if pattern3_mask.any():
        extracted = floor_series[pattern3_mask].str.extract(r'(\d+)\s*/\s*(\d+)\s*层')
        current_floors = extracted[0].astype(float)
        total_floors = extracted[1].astype(float)

        # Convert numeric floors into categorical descriptions
        def number_to_category(current_floor, total_floor):
            if pd.isna(current_floor) or pd.isna(total_floor) or total_floor == 0:
                return pd.NA
            ratio = current_floor / total_floor
            if current_floor == 1:
                return '底层'
            elif current_floor == total_floor:
                return '顶层'
            elif ratio <= 0.3:
                return '低楼层'
            elif ratio <= 0.7:
                return '中楼层'
            else:
                return '高楼层'

        categories = [number_to_category(cf, tf) for cf, tf in zip(current_floors, total_floors)]
        df.loc[pattern3_mask, 'current_floor'] = categories
        df.loc[pattern3_mask, 'total_floor'] = total_floors.astype('Int32')

    # pattern4: "地下1层", "B1层"
    pattern4_mask = ~pattern1_mask & ~pattern2_mask & ~pattern3_mask & floor_series.str.contains(
        r'^(?:地下\s*\d+\s*层|[Bb]\s*\d+\s*层)$', na=False
    )
    if pattern4_mask.any():
        df.loc[pattern4_mask, 'current_floor'] = '地下室'

    # Map Chinese floor names to English
    floor_mapping = {
        '地下室': 'basement',
        '底层': 'ground',
        '低楼层': 'low',
        '中楼层': 'medium',
        '高楼层': 'high',
        '顶层': 'top'
    }
    df['current_floor'] = df['current_floor'].map(floor_mapping)

    # Drop the original column
    df.drop(columns=[floor_column], inplace=True)

    # Summarize results
    processed_mask = df['current_floor'].notna()
    processed_count = processed_mask.sum()
    total_count = len(df)

    print(f"Processed {processed_count}/{total_count} rows successfully")
    unprocessed_count = total_count - processed_count
    if unprocessed_count > 0:
        print(f"Warning: {unprocessed_count} rows could not be processed")

        if verbose:
            print("\nUnprocessed entries:")
            print(df.loc[~processed_mask, :].assign(original=floor_series[~processed_mask])[['original']])


### elevator_clean

In [22]:
def elevator_house_ratio_clean(df: pd.DataFrame, ratio_column: str = '梯户比例') -> None:

    """
    Clean the elevator-household ratio column in-place.
    Parse strings like "一梯三户", "两梯十一户" into a float representing households per elevator.
    Overwrites df[ratio_column] with float values or None when parsing fails.
    """
    if ratio_column not in df.columns:
        raise KeyError(f"Column '{ratio_column}' not found in DataFrame.")

    pattern = re.compile(
        r'(?P<elev>[\d零一二三四五六七八九十两百千万亿]+)\s*梯\s*(?P<house>[\d零一二三四五六七八九十两百千万亿]+)\s*户'
    )

    def parse_entry(val) -> Optional[float]:
        if pd.isna(val):
            return None
        s = str(val).strip()
        if not s:
            return None

        m = pattern.search(s)
        if m:
            elev_raw = m.group('elev')
            house_raw = m.group('house')
            elev_num = int(elev_raw) if re.fullmatch(r'\d+', elev_raw) else translate_chinese_to_number(elev_raw)
            house_num = int(house_raw) if re.fullmatch(r'\d+', house_raw) else translate_chinese_to_number(house_raw)
            if elev_num is None or house_num is None or elev_num == 0:
                return None
            return float(house_num) / float(elev_num)

        tokens = re.findall(r'[\d零一二三四五六七八九十两百千万亿]+', s)
        if len(tokens) >= 2:
            elev_token, house_token = tokens[0], tokens[1]
            elev_num = int(elev_token) if re.fullmatch(r'\d+', elev_token) else translate_chinese_to_number(elev_token)
            house_num = int(house_token) if re.fullmatch(r'\d+', house_token) else translate_chinese_to_number(house_token)
            if elev_num is None or house_num is None or elev_num == 0:
                return None
            return float(house_num) / float(elev_num)

        return None

    # overwrite column in-place; pandas will store None as NaN
    df.loc[:, ratio_column] = df[ratio_column].apply(parse_entry)



### direction_clean

In [23]:
def direction_clean(data: pd.DataFrame, column: str = "朝向") -> None:
    """
    Clean orientation strings in-place.
    Parse direction tokens and reorder them according to:
      东 > 东南 > 南 > 西南 > 西 > 西北 > 北 > 东北
    The final value will be joined by '-' (e.g. "南-西南-西").
    Unparsable entries become None (stored as NaN in the DataFrame).
    """
    if column not in data.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame.")

    # desired order (index defines sort priority)
    order_list = ['东', '东南', '南', '西南', '西', '西北', '北', '东北']
    order_map = {d: i for i, d in enumerate(order_list)}

    # pattern: match multi-char directions first (东南/西南/西北/东北), then single chars
    pattern = re.compile(r'(东南|西南|西北|东北|东|南|西|北)')

    def parse_and_sort(value) -> Optional[str]:
        if pd.isna(value):
            return None
        s = str(value).strip()
        if not s:
            return None

        # find all direction tokens in the string (in textual order)
        tokens = pattern.findall(s)
        if not tokens:
            return None

        # keep only valid tokens and deduplicate
        # use set for membership, but we want final order by order_map
        valid_tokens = {t for t in tokens if t in order_map}
        if not valid_tokens:
            return None

        # sort according to order_map and produce list
        sorted_tokens = sorted(valid_tokens, key=lambda t: order_map[t])

        # join with hyphen
        return "-".join(sorted_tokens)

    # apply and write back in a single assignment to avoid chained-assignment warnings
    data.loc[:, column] = data[column].apply(parse_and_sort)


### house_type_clean

In [24]:
def house_type_clean(data: pd.DataFrame, house_type_str: str) -> None:
    """
    Extract the number of bedrooms, living rooms, kitchens, and toilets from the house type string column
    and add corresponding columns directly to the original DataFrame.
    
    Parameters:
        data: DataFrame to be processed (modified in place)
        house_type_str: Column name storing house type strings (e.g., "house_type")
    """
    
    def _parse_single_house_type(house_str):
        """Internal helper function: Parse single house type string and return counts for bedroom, living room, kitchen, toilet"""
        # 1. Handle non-string/NaN/invalid values (e.g., "·", "garage")
        if pd.isna(house_str) or not isinstance(house_str, str):
            return np.nan, np.nan, np.nan, np.nan
        
        # Define meaningless invalid strings
        invalid_values = {'·', '车库'}
        # Return all NaN if string is invalid, or contains no key info and no "unknown"
        has_valid_key = any(key in house_str for key in ['室', '厅', '厨', '卫', '居室', '房间'])
        if house_str in invalid_values or (not has_valid_key and '未知' not in house_str):
            return np.nan, np.nan, np.nan, np.nan
        
        # 2. Define extraction rules for each field (regex + special handling)
        # Bedroom: match "number + 室/居室/房间", return NaN if contains "unknown room"
        if '未知室' in house_str:
            bedroom = np.nan
        else:
            bedroom_match = re.search(r'(\d+)[室居室房间]', house_str)
            bedroom = int(bedroom_match.group(1)) if bedroom_match else 0  # Default to 0 if not matched
        
        # Living room: match "number + 厅", return NaN if contains "unknown living room"
        if '未知厅' in house_str:
            living_room = np.nan
        else:
            living_room_match = re.search(r'(\d+)厅', house_str)
            living_room = int(living_room_match.group(1)) if living_room_match else 0
        
        # Kitchen: match "number + 厨", return NaN if contains "unknown kitchen"
        if '未知厨' in house_str:
            kitchen = np.nan
        else:
            kitchen_match = re.search(r'(\d+)厨', house_str)
            kitchen = int(kitchen_match.group(1)) if kitchen_match else 0
        
        # Toilet: match "number + 卫", return NaN if contains "unknown toilet"
        if '未知卫' in house_str:
            toilet = np.nan
        else:
            toilet_match = re.search(r'(\d+)卫', house_str)
            toilet = int(toilet_match.group(1)) if toilet_match else 0
        
        return bedroom, living_room, kitchen, toilet
    
    # 3. Batch parse the house type column and add "bedroom, living_room, kitchen, toilet" columns to original DataFrame
    # Apply parsing function to each element in house type column, return 4 columns of data
    parsed_results = data[house_type_str].apply(
        lambda x: pd.Series(_parse_single_house_type(x), index=['bedroom', 'living_room', 'kitchen', 'toilet'])
    )
    # Merge parsing results into original DataFrame (directly modify original data)
    for col in parsed_results.columns:
        data[col] = parsed_results[col]

### delete columns

In [25]:
def delete_columns(
    data: pd.DataFrame,
    exam: pd.DataFrame,
    cols_to_delete: List[str],
    delete_text: bool = False,
    delete_single_value: bool = False,
    y_str: Optional[str] = None,
    label_str: Optional[str] = None,
    delete_unique_columns: bool = False
) -> None:
    """
    Delete columns from training and exam DataFrames in-place.
    Strict deletion:
      - Whole-column single value
      - Columns where non-NaN values are all identical (subset single-value)
      - Optionally delete text/non-numeric columns
      - Optionally delete columns that appear in only one DataFrame

    Parameters
    ----------
    data : pd.DataFrame
        Training DataFrame to modify in-place.
    exam : pd.DataFrame
        Exam/validation DataFrame to modify in-place.
    cols_to_delete : list of str
        Columns that must be deleted.
    delete_text : bool
        If True, delete all text/non-numeric columns in training set.
    delete_single_value : bool
        If True, delete all single-value columns in training set (even non-NaN subset)
    y_str : str, optional
        Column name to protect from deletion (e.g., target variable).
    label_str : str, optional
        Column name to protect from deletion (e.g., ID column).
    delete_unique_columns : bool
        If True, delete columns that appear in only one DataFrame (data or exam).
    """
    
    # Create protected columns set
    protected_cols = set()
    if y_str is not None:
        protected_cols.add(y_str)
    if label_str is not None:
        protected_cols.add(label_str)
    
    # Initialize drop set with specified columns, excluding protected ones
    to_drop = set(col for col in cols_to_delete if col not in protected_cols)

    # Step 1: check whole-column single value
    if delete_single_value:
        single_value_cols = [col for col in data.columns 
                            if col not in protected_cols and data[col].nunique(dropna=True) <= 1]
        to_drop.update(single_value_cols)

        # Step 1b: check non-NaN subset single value
        for col in data.columns:
            if col in protected_cols:
                continue
            non_na_values = data[col].dropna()
            if len(non_na_values) > 0 and non_na_values.nunique() == 1:
                to_drop.add(col)

    # Step 2: check text columns
    if delete_text:
        text_cols = [col for col in data.columns 
                    if col not in protected_cols and not pd.api.types.is_numeric_dtype(data[col])]
        to_drop.update(text_cols)

    # Step 3: delete columns that appear in only one DataFrame
    if delete_unique_columns:
        # Count column occurrences across both DataFrames
        all_cols = list(data.columns) + list(exam.columns)
        col_counts = Counter(all_cols)
        
        # Find columns that appear only once (in either data or exam)
        unique_cols = [col for col, count in col_counts.items() 
                      if count == 1 and col not in protected_cols]
        to_drop.update(unique_cols)
        

    # Step 4: erase both data and exam columns
    for df in (data, exam):
        # Only drop columns that exist in the DataFrame
        cols_in_df = [col for col in to_drop if col in df.columns]
        if cols_in_df:
            df.drop(columns=cols_in_df, inplace=True)

### to_numpy : this function contains z-score function

In [26]:
def change_to_numpy(
    data: pd.DataFrame,
    exam: pd.DataFrame,
    label_str: Optional[str],
    y_str: str,
    verbose: bool = True
) -> Tuple[np.ndarray, np.ndarray, float, float, np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Returns:
      X_train_imputed, y_train (standardized), y_mean, y_std,
      X_exam_imputed, exam_ids, exam_y (may be None)
    """
    import numpy as np
    import pandas as pd
    from sklearn.impute import SimpleImputer

    # Work on copies
    df = data.copy()
    ex = exam.copy()

    # exam ids
    exam_ids = None
    if label_str is not None and label_str in ex.columns:
        exam_ids = ex[label_str].to_numpy(copy=True)

    # exam y (if present)
    exam_y = None
    if y_str in ex.columns:
        exam_y_series = pd.to_numeric(ex[y_str], errors='coerce')
        if exam_y_series.notna().any():
            exam_y = exam_y_series.to_numpy(copy=True)
            if verbose:
                print(f"[change_to_numpy] exam_y extracted, shape={exam_y.shape}")
        else:
            exam_y = None

    # drop rows in training with missing y
    df[y_str] = pd.to_numeric(df[y_str], errors='coerce')
    df = df[df[y_str].notna()].reset_index(drop=True)
    if verbose:
        print(f"[change_to_numpy] training rows after dropping missing y: {len(df)}")

    # select numeric/boolean features excluding label and target
    candidate_cols = df.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    for c in (y_str, label_str):
        if c in candidate_cols:
            candidate_cols.remove(c)
    if len(candidate_cols) == 0:
        raise ValueError("No numeric/boolean candidate features available.")

    feature_cols = [c for c in df.columns if c in candidate_cols]

    # ensure exam has same feature columns
    for c in feature_cols:
        if c not in ex.columns:
            ex[c] = np.nan
    ex = ex.reindex(columns=feature_cols)

    # convert to float64
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float64)
        ex[col] = pd.to_numeric(ex[col], errors='coerce').astype(np.float64)

    # impute missing values using training mean
    imp = SimpleImputer(strategy='mean')
    X_train = df[feature_cols].to_numpy(copy=True)
    X_exam = ex[feature_cols].to_numpy(copy=True)
    data_x = imp.fit_transform(X_train)
    exam_x = imp.transform(X_exam)

    # standardize y
    y_series = pd.to_numeric(df[y_str], errors='coerce')
    data_y_mean = float(y_series.mean())
    data_y_std = float(y_series.std(ddof=0))
    if data_y_std == 0 or np.isnan(data_y_std):
        data_y = (y_series - data_y_mean).to_numpy(copy=True)
    else:
        data_y = ((y_series - data_y_mean) / data_y_std).to_numpy(copy=True)

    if verbose:
        print(f"[change_to_numpy] finished. train={data_x.shape}, exam={exam_x.shape}")

    return data_x, data_y, data_y_mean, data_y_std, exam_x, exam_ids, exam_y


# package

## type 1 function

这个部分的函数是调用直接处理data和exam的函数,不需要出现"用训练集的参数调整测试集"

In [27]:
def type1function(data_set,floor_str,elevator_exist_str,water_str,electricity_str,heat_str,house_type_str,label_str,y_str,main_index):
    print("deleted columns:")
    erase_missing_data(data_set[0], data_set[1], column_ratio=0.8, row_ratio=34/54)

    # data cleaning
    for i in range(0,2):

        # eraze identical data
        merge_identical_column(data_set[i],'lon','coord_x','average')
        merge_identical_column(data_set[i],'lat','coord_y','average')
        merge_identical_column(data_set[i],'区县','区域','select')
        merge_identical_column(data_set[i],'板块','板块_comm','select')
        merge_identical_column(data_set[i],'环线','环线位置','select')
        data_set[i].rename(columns={'区县': 'county'}, inplace=True)
        data_set[i].rename(columns={'板块': 'plate'}, inplace=True)
        data_set[i].rename(columns={'环线': 'ring'}, inplace=True)

        merge_identical_column(data_set[i],'供水','用水','select')
        merge_identical_column(data_set[i],'供电','用电','select')
        merge_identical_column(data_set[i],'供暖','采暖','select')
        data_set[i].rename(columns={'供水':water_str}, inplace=True)
        data_set[i].rename(columns={'供电':electricity_str}, inplace=True)
        data_set[i].rename(columns={'供暖':heat_str}, inplace=True)

        # these data share identical feature: number+unit
        first_numeric_clean(data_set[i],'建筑面积')
        first_numeric_clean(data_set[i],'套内面积')
        first_numeric_clean(data_set[i],'面积')
        first_numeric_clean(data_set[i],'房屋总数')
        first_numeric_clean(data_set[i],'楼栋总数')
        first_numeric_clean(data_set[i],'绿 化 率')

        # these data share identical feature: min-max+unit or number+unit
        numeric_range_clean(data_set[i],'物 业 费',method='medium')
        numeric_range_clean(data_set[i],'燃气费',method='medium')
        numeric_range_clean(data_set[i],'供热费',method='medium')
        numeric_range_clean(data_set[i],'建筑年代',method='medium')    # I give up split
        
        floor_clean(data_set[i], floor_str)
        house_type_clean(data_set[i],house_type_str)
        
        normalize_substr(data_set[i], '付款方式', '付价', mode='contain')    # because it contains unexpected 'http://'
        if task[main_index] == "price": 
            elevator_house_ratio_clean(data_set[i])    

        # why we eraze '套内面积'
        if task[main_index] == "price":
            print("="*30)
            print(f"{str0[i]} data report: ")
            print("="*30)
            linear_correlation_analysis(data_set[i],'建筑面积','套内面积')
            if '套内面积' in data_set[i].columns:
                data_set[i].drop(columns=['套内面积'], inplace=True)

    if task[main_index] == "price":
        temp = "建筑面积"
    else:
        temp = "面积"
    for i in range(0,len(data_set)):
        if temp in data_set[i].columns:
            data_set[i].rename(columns={temp: 'area'}, inplace=True)

## type 2 function
If not, data and exam may treat separately, which is a mistake of viewing test set.  
(type1 means separately treating does not affect result)

In [28]:
def type2function(data,exam,elevator_exist_str:str,water_str:str,electricity_str:str,heat_str:str,house_type_str:str,label_str:str,y_str:str,direction_str,furnish_str,main_index,silent):
    knn_fill_categorical(data,exam)
    print("knn_fill_categorical completed.\n")
    
    # because '建筑结构' '建筑结构_comm' has different name
    if task[main_index] == "price":
        fill_missing(data, exam, '建筑结构', 'naive', '未知结构',verbose = not silent)
        # Because in training data, 未知结构 already exists
        # so fill in string '未知结构' is reasonable.
        to_dummy_multiple(data,exam,['建筑结构'],silent=silent)
        extract_dummy_substr_split(data,exam,'建筑结构_comm','/',silent=silent)
    else:
        extract_dummy_substr_split(data,exam,'建筑结构','/',silent=silent)

    fill_missing(data, exam, '装修情况', 'naive', '未知',verbose=not silent)
    fill_missing(data, exam, elevator_exist_str, 'naive', value='unknown',verbose=not silent)

    # extract substr
    # direction_clean(data_set[i], direction_str) : because Dimensional disaster
    substr_to_extract_split = ['产权描述',water_str,electricity_str,heat_str,'房屋优势','物业类别',direction_str]
    delimiter = ['/','/','/','/','、','/',' ']
    for j in range(0,len(substr_to_extract_split)):
        extract_dummy_substr_split(data,exam,substr_to_extract_split[j],delimiter[j],silent=silent)

    # delete single_value columns to avoid perfect multicollinearity (because it equal to intercept)
    to_dummy_multiple(data, exam,['交易权属','产权所属','房屋用途','current_floor','ring',elevator_exist_str,'房屋年限',direction_str,furnish_str,'租赁方式','城市'],silent=silent)
    column_to_delete = ['物业办公电话','lon','lat','plate','county',house_type_str,'开发商','物业公司']
    delete_columns(data,exam,column_to_delete,label_str='ID',y_str='Price',delete_text=True,delete_single_value=True,delete_unique_columns=True)

    # re-eraze missing data
    erase_missing_data(data,exam,0.5,1)
    # take the logarithm of right-skewed data
    log_y = log_transform_skewed(data,exam,y_str=y_str,label_str=label_str)
    fill_missing(data, exam, '停车位', 'ols', condition_column='房屋总数',verbose=not silent)
    
    numeric_columns = data.select_dtypes(include=[np.number]).columns.tolist()
    protected_columns = []
    if y_str is not None and y_str in numeric_columns:
        protected_columns.append(y_str)
    if label_str is not None and label_str in numeric_columns:
        protected_columns.append(label_str)

    columns_to_fill = [col for col in numeric_columns if col not in protected_columns]

    # fill all missing columns(except protected columns)
    for column in columns_to_fill:
        if column in data.columns and (data[column].isna().any() or exam[column].isna().any()):
            print(f"Filling {column} with average method...")
            fill_missing(data, exam, column, 'average',verbose = not silent)

    print("Missing value filling completed!")

    # process outliers
    for col in data.columns:
        if col in exam.columns:
            outlier_process(data,exam,col,verbose=not silent)
    if not silent:
        print("outlier process completed.\n")

    # VIF(also containing linear correlation analysis) to perform feature selection
    VIF_process(data,exam,perform_delete=True,silent=silent)
    if not silent:
        print("VIF_process completed.\n")
        
    # finally, generate numpy from pandas
    data_x, data_y, data_y_mean, data_y_std, exam_x,exam_id,exam_y=change_to_numpy(data,exam,label_str,y_str,verbose = not silent)
    if not silent:
        print("pandas has changed to numpy.\n")
    return data_x,data_y,data_y_mean,data_y_std,exam_x,exam_id,exam_y,log_y

# analysis data

## load data

In [29]:
data = pd.DataFrame()
exam = pd.DataFrame()
y_str = 'Price'
label_str = 'ID'
main_index = 0
# load data for exam
data,exam = final_load_data(main_index,task)
data_set = [data, exam]

# in price and rent, name is slightly different
# here we control the name
house_type_str = floor_str = elevator_exist_str = furnish_str = direction_str = ""
heat_str = 'heat_supply'
electricity_str = 'electricity_supply'
water_str = 'water_supply'

if task[main_index] == "price":
    direction_str = '房屋朝向'
    furnish_str = '装修情况'
    elevator_exist_str = '配备电梯'
    floor_str = '所在楼层'
    house_type_str = '房屋户型'
elif task[main_index] == "rent":
    direction_str = '朝向'
    furnish_str = '装修'
    elevator_exist_str = '电梯'
    floor_str = '楼层'
    house_type_str = '户型'

expected_str = ['城市','区县','区域','板块','板块_comm']
for i in range(0,1):
    for col in expected_str:
        if col in data_set[i].columns:
            data_set[i][col] = data_set[i][col].astype('Int32').astype('string')

Success: exact match found: ruc_Class25Q2_test_price.csv
Success: exact match found: .\ruc_Class25Q2_test_price.csv
Trying encoding: utf-8-sig


C:\Users\lumine\AppData\Local\Temp\ipykernel_5500\360693213.py:151: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding=encoding)


Successfully read .\ruc_Class25Q2_test_price.csv with utf-8-sig encoding
Success: find price for test
Success: exact match found: ruc_Class25Q2_train_price.csv
Success: exact match found: .\ruc_Class25Q2_train_price.csv
Trying encoding: utf-8-sig


C:\Users\lumine\AppData\Local\Temp\ipykernel_5500\360693213.py:151: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding=encoding)


Successfully read .\ruc_Class25Q2_train_price.csv with utf-8-sig encoding
Success: find price for train
Data is ready!


## data analysis

### missing data

In [30]:
# view the missing data situation
if debug_mode:
    for i in range(0,len(str0)):
        print("="*30)
        print(f"{str0[i]} data report: ")
        print("="*30)
        data_missing_report(data_set[i],False, True)

### convert to set

In [31]:
if debug_mode:
    price_name, price_set = transfer_to_set(data)
    price_name_exam, price_set_exam = transfer_to_set(exam)

In [32]:
if debug_mode:
    object = "物业类别"
    for i in range(0, len(price_name)):
        if object in price_name[i]:
            print(f"{object} size: {len(price_set[i])} ,{price_set[i]}")

### identical column check

In [33]:
if debug_mode:
    matrix = [['lon','coord_x'],['lat','coord_y'],['区县','区域'],['板块','板块_comm'],['环线','环线位置'],['供水','用水'],['供电','用电'],['供暖','采暖']]
    for i in range(0,len(data_set)):
        for j in range(0,len(matrix)):
            if matrix[j][0] in data_set[i].columns and matrix[j][1] in data_set[i].columns:
                identical_column_test(data_set[i],matrix[j][0],matrix[j][1],5)
            else:
                print(f"There exists no duplicate between {matrix[j][0]} and {matrix[j][1]}")
        print(f"loop for {str0[i]} ends. \n\n\n")

the output is, lon and coord_x, lat and coord_y is quite close to each other. Therefore, it's reasonable to guess that small differences result from measurement mistakes.  
However, differences between '板块','板块_comm' '环线','环线位置','区域','区县' is quite different.
Moreover, relative accurate coord_x and coord_y itself is not quite helpful in predicting  prices: compared with coord_x and coord_y, the administrative division and Six Ring Line is more helpful in prediction price.  
Conclusion:  it's better to consider using coord_x and coord_y and KNN method to fill the missing data of 环线, 区域, etc.

### region relation

#### hierarchy_validation

In [34]:
if debug_mode:
    vector = ['城市', 'county', 'ring', 'plate']
    relation_df = pd.DataFrame(index=vector, columns=vector)

    for parent in vector:
        for child in vector:
            if parent == child:
                relation_df.loc[parent, child] = 1
            else:
                result = probabilistic_hierarchy_validation(data, parent, child, 0.90, False)
                relation_df.loc[parent, child] = result

    print(relation_df)
    # result: city->county->plate, therefore, delete city and county

#### ANOVA Analysis

In [35]:
if debug_mode:
    two_way_ANOVA(data, county_col='county', community_col='plate', 
              area_col='area', price_col='Price', min_samples=40)

# main_index = 0 (price)

## load data

In [36]:
main_index = 0
data,exam = final_load_data(main_index,task)
data_set= [data, exam]
house_type_str = floor_str = elevator_exist_str = furnish_str = direction_str = ""
heat_str = 'heat_supply'
electricity_str = 'electricity_supply'
water_str = 'water_supply'

if task[main_index] == "price":
    direction_str = '房屋朝向'
    furnish_str = '装修情况'
    elevator_exist_str = '配备电梯'
    floor_str = '所在楼层'
    house_type_str = '房屋户型'
elif task[main_index] == "rent":
    direction_str = '朝向'
    furnish_str = '装修'
    elevator_exist_str = '电梯'
    floor_str = '楼层'
    house_type_str = '户型'

expected_str = ['城市','区县','区域','板块','板块_comm']
for i in range(0,1):
    for col in expected_str:
        if col in data_set[i].columns:
            data_set[i][col] = data_set[i][col].astype('Int32').astype('string')

Success: exact match found: ruc_Class25Q2_test_price.csv
Success: exact match found: .\ruc_Class25Q2_test_price.csv
Trying encoding: utf-8-sig


C:\Users\lumine\AppData\Local\Temp\ipykernel_5500\360693213.py:151: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding=encoding)


Successfully read .\ruc_Class25Q2_test_price.csv with utf-8-sig encoding
Success: find price for test
Success: exact match found: ruc_Class25Q2_train_price.csv
Success: exact match found: .\ruc_Class25Q2_train_price.csv
Trying encoding: utf-8-sig


C:\Users\lumine\AppData\Local\Temp\ipykernel_5500\360693213.py:151: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding=encoding)


Successfully read .\ruc_Class25Q2_train_price.csv with utf-8-sig encoding
Success: find price for train
Data is ready!


In [37]:
type1function(data_set,floor_str,elevator_exist_str,water_str,electricity_str,heat_str,house_type_str,label_str,y_str,main_index)

deleted columns:
  - 别墅类型: Train(98.6%) Exam(99.5%)
  - 抵押信息: Train(100.0%) Exam(100.0%)
Averaged 4034 conflicting numeric values between lon and coord_x
For 5280 non-numeric conflicts, kept existing values from lon (equal/better missing rate).
Deleted column: coord_x
Final missing values in lon: 0 (0.00%)
Averaged 3392 conflicting numeric values between lat and coord_y
For 5914 non-numeric conflicts, kept existing values from lat (equal/better missing rate).
Deleted column: coord_y
Final missing values in lat: 0 (0.00%)
Filled 6901 missing values in 区县 from 区域
For 1744 conflicts, chose values from 区域 (lower missing rate).
Deleted column: 区域
Final missing values in 区县: 0 (0.00%)
For 1891 conflicts, kept existing values from 板块 (equal/better missing rate).
Deleted column: 板块_comm
Final missing values in 板块: 0 (0.00%)
For 59 conflicts, kept existing values from 环线 (equal/better missing rate).
Deleted column: 环线位置
Final missing values in 环线: 63114 (60.96%)
Error: One or both columns not f

## train and test model

In [38]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
random_seed = 111

# 1. 数据分割
pre_train, test = train_test_split(data, test_size=0.2, random_state=random_seed)
pre_train = pre_train.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"原始数据形状: {data.shape}")
print(f"预训练集形状: {pre_train.shape}")
print(f"测试集形状: {test.shape}")

# 2. 数据预处理
X_pre_train, y_pre_train, y_pre_train_mean, y_pre_train_std, test_x, test_id, test_y, log_y = type2function(
    pre_train, test, elevator_exist_str, water_str, electricity_str, heat_str, house_type_str, label_str, y_str, direction_str,furnish_str,main_index,silent
)

print(f"\n=== 数据预处理完成 ===")
print(f"训练集: {X_pre_train.shape}, 测试集: {test_x.shape}")
print(f"log_y: {log_y}, y_mean: {y_pre_train_mean:.4f}, y_std: {y_pre_train_std:.4f}")

# 3. 定义反标准化函数
def inverse_transform(y_standardized, mean, std, log_y):
    """将标准化值转换回原始尺度"""
    y = y_standardized * std + mean
    if log_y:
        y = np.exp(y)
    return y

# 4. 定义性能评估函数
def evaluate_model_performance(model, X_train, y_train_std, X_test, y_test_original, 
                              mean, std, log_y, model_name):
    """评估模型性能并返回MAE和RMAE"""
    
    # 训练集预测和反标准化
    y_train_pred_std = model.predict(X_train)
    y_train_true = inverse_transform(y_train_std, mean, std, log_y)
    y_train_pred = inverse_transform(y_train_pred_std, mean, std, log_y)
    
    # 测试集预测和反标准化
    y_test_pred_std = model.predict(X_test)
    y_test_pred = inverse_transform(y_test_pred_std, mean, std, log_y)
    y_test_true = y_test_original
    
    # 计算MAE
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    test_mae = mean_absolute_error(y_test_true, y_test_pred)
    
    # 计算RMAE (MAE / mean(|y_true|))
    train_rmae = train_mae / np.mean(np.abs(y_train_true))
    test_rmae = test_mae / np.mean(np.abs(y_test_true))
    
    # 交叉验证性能
    cv_scores = cross_val_score(model, X_train, y_train_std, 
                               cv=6, scoring='neg_mean_absolute_error')
    cv_mae = -np.mean(cv_scores)
    
    # 交叉验证的RMAE估计
    y_cv_pred_std = cross_val_predict(model, X_train, y_train_std, cv=6)
    y_cv_pred = inverse_transform(y_cv_pred_std, mean, std, log_y)
    y_cv_true = inverse_transform(y_train_std, mean, std, log_y)
    cv_rmae = mean_absolute_error(y_cv_true, y_cv_pred) / np.mean(np.abs(y_cv_true))
    
    print(f"\n{model_name} 性能:")
    print(f"  训练集 MAE: {train_mae:.2f}, RMAE: {train_rmae:.4f}")
    print(f"  测试集 MAE: {test_mae:.2f}, RMAE: {test_rmae:.4f}")
    print(f"  交叉验证 RMAE: {cv_rmae:.4f}")
    
    return {
        'train_mae': train_mae,
        'train_rmae': train_rmae,
        'test_mae': test_mae,
        'test_rmae': test_rmae,
        'cv_rmae': cv_rmae
    }

# 5. train model
print(f"\n=== 开始模型训练和调优 ===")

# 5.1 OLS模型
print("训练OLS模型...")
ols = LinearRegression()
ols.fit(X_pre_train, y_pre_train)
ols_best_params = "No hyperparameters (baseline)"

# 5.2 Lasso回归
print("调优Lasso模型...")
lasso = Lasso(random_state=random_seed, max_iter=10000)
lasso_param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
lasso_grid = GridSearchCV(
    lasso, lasso_param_grid, 
    cv=6,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
lasso_grid.fit(X_pre_train, y_pre_train)
best_lasso = lasso_grid.best_estimator_
lasso_best_params = lasso_grid.best_params_

# 5.3 岭回归
print("调优Ridge模型...")
ridge = Ridge(random_state=random_seed)
ridge_param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(
    ridge, ridge_param_grid,
    cv=6,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
ridge_grid.fit(X_pre_train, y_pre_train)
best_ridge = ridge_grid.best_estimator_
ridge_best_params = ridge_grid.best_params_

# 5.4 ElasticNet
print("调优ElasticNet模型...")
elastic_net = ElasticNet(random_state=random_seed, max_iter=10000)
en_param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
en_grid = GridSearchCV(
    elastic_net, en_param_grid,
    cv=6,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
en_grid.fit(X_pre_train, y_pre_train)
best_en = en_grid.best_estimator_
en_best_params = en_grid.best_params_


best_params_dict = {
    'OLS': ols_best_params,
    'LASSO': lasso_best_params,
    'Ridge': ridge_best_params,
    'ElasticNet': en_best_params
}


print(f"\n=== best params ===")
for model_name, params in best_params_dict.items():
    print(f"{model_name}: {params}")

# 6. check all model's performance


models = {
    'OLS': ols,
    'LASSO': best_lasso,
    'Ridge': best_ridge,
    'ElasticNet': best_en
}

performance_results = {}
for name, model in models.items():
    performance_results[name] = evaluate_model_performance(
        model, X_pre_train, y_pre_train, test_x, test_y,
        y_pre_train_mean, y_pre_train_std, log_y, name
    )

# 7. find best model
best_model_name = min(performance_results.items(), 
                     key=lambda x: x[1]['test_mae'])[0]
print(f"\n=== best model: {best_model_name} ===")

# 8. performance
print(f"\n=== 最终性能表格 ===")
print("| Metrics | In sample (RMAE) | Out of sample (RMAE) | Cross-validation (RMAE) | Best Hyperparameters | Kaggle Score(模拟值) |")
print("|---------|------------------|----------------------|------------------------|---------------------|--------------|")

# 模拟Kaggle分数（实际中需要提交到Kaggle平台）
kaggle_scores = {
    'OLS': 60,
    'LASSO': 61,
    'Ridge': 62,
    'ElasticNet': 61
}

for model_name in ['OLS', 'LASSO', 'Ridge', 'ElasticNet']:
    metrics = performance_results[model_name]
    params = best_params_dict[model_name]
    kaggle_score = kaggle_scores.get(model_name, 60)
    print(f"| {model_name} | {metrics['train_rmae']:.4f} | {metrics['test_rmae']:.4f} | {metrics['cv_rmae']:.4f} | {params} | {kaggle_score} |")

# best model
best_metrics = performance_results[best_model_name]
best_params = best_params_dict[best_model_name]
best_kaggle = kaggle_scores.get(best_model_name, 62)
print(f"| Best Linear Model ({best_model_name}) | {best_metrics['train_rmae']:.4f} | {best_metrics['test_rmae']:.4f} | {best_metrics['cv_rmae']:.4f} | {best_params} | {best_kaggle} |")



原始数据形状: (103531, 52)
预训练集形状: (82824, 52)
测试集形状: (20707, 52)
knn_fill_categorical completed.

Warning: Column '房屋年限' has 35271 missing in training, 8922 in exam
  - 供热费: Train(69.6%) Exam(70.1%)
Target 'Price' log-transformed: skew 4.3206 -> 0.2617
  Used OLS regression: y = 0.7287 * x + 1.1968
  R² score on training data: 0.2956
Filling 建筑年代 with average method...
  Used numeric mean value from data: 2006.619967
Filling 房屋总数 with average method...
  Used numeric mean value from data: 7.090105
Filling 楼栋总数 with average method...
  Used numeric mean value from data: 2.684413
Filling 绿 化 率 with average method...
  Used numeric mean value from data: 3.437103
Filling 容 积 率 with average method...
  Used numeric mean value from data: 0.889405
Filling 物 业 费 with average method...
  Used numeric mean value from data: 0.633952
Filling 燃气费 with average method...
  Used numeric mean value from data: 2.683969
Filling bedroom with average method...
  Used numeric mean value from data: 2.614448
Fill

## Regression: use all data

### preparation

In [39]:
data_x,data_y,data_y_mean,data_y_std,exam_x,exam_id,exam_y,log_y = type2function(data,exam,elevator_exist_str,water_str,electricity_str,heat_str,house_type_str,label_str,y_str,direction_str,furnish_str,main_index,silent)

knn_fill_categorical completed.

Warning: Column '房屋年限' has 44193 missing in training, 11296 in exam
  - 供热费: Train(69.7%) Exam(67.7%)
Target 'Price' log-transformed: skew 4.3484 -> 0.2546
  Used OLS regression: y = 0.7243 * x + 1.2305
  R² score on training data: 0.2957
Filling 建筑年代 with average method...
  Used numeric mean value from data: 2006.636688
Filling 房屋总数 with average method...
  Used numeric mean value from data: 7.089468
Filling 楼栋总数 with average method...
  Used numeric mean value from data: 2.683643
Filling 绿 化 率 with average method...
  Used numeric mean value from data: 3.438234
Filling 容 积 率 with average method...
  Used numeric mean value from data: 0.889059
Filling 物 业 费 with average method...
  Used numeric mean value from data: 0.634611
Filling 燃气费 with average method...
  Used numeric mean value from data: 2.683228
Filling bedroom with average method...
  Used numeric mean value from data: 2.613264
Filling living_room with average method...
  Used numeric mean 

### OLS

In [40]:
from sklearn.linear_model import LinearRegression

# 假设 log_y 是布尔值，表示 y 是否被 log-transform
# y_std, y_mean 已在标准化过程中计算
# x, y, exam_final 已准备好

model = LinearRegression(fit_intercept=True)
model.fit(data_x, data_y)

# 系数与截距
beta = model.coef_
intercept = model.intercept_

# predict_data
y_pred = model.predict(exam_x)

y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

import pandas as pd
y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_OLS.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_OLS.csv", index=False, mode='a', header=False)


### LASSO

In [41]:
# LASSO模型
from sklearn.linear_model import Lasso

# 使用字典中的最佳参数
lasso_params = best_params_dict['LASSO']
model = Lasso(**lasso_params, random_state=111, max_iter=10000)
model.fit(data_x, data_y)

# 预测
y_pred = model.predict(exam_x)
y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

import pandas as pd
y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_LASSO.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_LASSO.csv", index=False, mode='a', header=False)

### Ridge

In [42]:
# Ridge模型
from sklearn.linear_model import Ridge

# 使用字典中的最佳参数
ridge_params = best_params_dict['Ridge']
model = Ridge(**ridge_params, random_state=111)
model.fit(data_x, data_y)

# 预测
y_pred = model.predict(exam_x)
y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_Ridge.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_Ridge.csv", index=False, mode='a', header=False)

### ElasticNet

In [43]:
# ElasticNet模型

from sklearn.linear_model import ElasticNet

# 使用字典中的最佳参数
en_params = best_params_dict['ElasticNet']
model = ElasticNet(**en_params, random_state=111, max_iter=10000)
model.fit(data_x, data_y)

# 预测
y_pred = model.predict(exam_x)
y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_ElasticNet.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_ElasticNet.csv", index=False, mode='a', header=False)

# main index = 1 (rent)

## load data

In [44]:
main_index = 1
data,exam = final_load_data(main_index,task)
data_set= [data, exam]
house_type_str = floor_str = elevator_exist_str = furnish_str = direction_str = ""
heat_str = 'heat_supply'
electricity_str = 'electricity_supply'
water_str = 'water_supply'

if task[main_index] == "price":
    direction_str = '房屋朝向'
    furnish_str = '装修情况'
    elevator_exist_str = '配备电梯'
    floor_str = '所在楼层'
    house_type_str = '房屋户型'
elif task[main_index] == "rent":
    direction_str = '朝向'
    furnish_str = '装修'
    elevator_exist_str = '电梯'
    floor_str = '楼层'
    house_type_str = '户型'

expected_str = ['城市','区县','区域','板块','板块_comm']
for i in range(0,1):
    for col in expected_str:
        if col in data_set[i].columns:
            data_set[i][col] = data_set[i][col].astype('Int32').astype('string')

Success: exact match found: ruc_Class25Q2_test_rent.csv
Success: exact match found: .\ruc_Class25Q2_test_rent.csv
Trying encoding: utf-8-sig
Successfully read .\ruc_Class25Q2_test_rent.csv with utf-8-sig encoding
Success: find rent for test
Success: exact match found: ruc_Class25Q2_train_rent.csv
Success: exact match found: .\ruc_Class25Q2_train_rent.csv
Trying encoding: utf-8-sig
Successfully read .\ruc_Class25Q2_train_rent.csv with utf-8-sig encoding
Success: find rent for train
Data is ready!


In [45]:
type1function(data_set,floor_str,elevator_exist_str,water_str,electricity_str,heat_str,house_type_str,label_str,y_str,main_index)

deleted columns:
Averaged 4645 conflicting numeric values between lon and coord_x
For 11274 non-numeric conflicts, kept existing values from lon (equal/better missing rate).
Deleted column: coord_x
Final missing values in lon: 0 (0.00%)
Averaged 6485 conflicting numeric values between lat and coord_y
For 9424 non-numeric conflicts, kept existing values from lat (equal/better missing rate).
Deleted column: coord_y
Final missing values in lat: 0 (0.00%)
Error: One or both columns not found: 区县, 区域
Error: One or both columns not found: 板块, 板块_comm
Error: One or both columns not found: 环线, 环线位置
Filled 11816 missing values in 供水 from 用水
For 25421 conflicts, chose values from 用水 (lower missing rate).
Deleted column: 用水
Final missing values in 供水: 6725 (6.99%)
Filled 12012 missing values in 供电 from 用电
For 25782 conflicts, chose values from 用电 (lower missing rate).
Deleted column: 用电
Final missing values in 供电: 6535 (6.79%)
Filled 2708 missing values in 供暖 from 采暖
For 3723 conflicts, kept exis

## train and test model

In [46]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
random_seed = 111

# 1. 数据分割
pre_train, test = train_test_split(data, test_size=0.2, random_state=random_seed)
pre_train = pre_train.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"原始数据形状: {data.shape}")
print(f"预训练集形状: {pre_train.shape}")
print(f"测试集形状: {test.shape}")

# 2. 数据预处理
X_pre_train, y_pre_train, y_pre_train_mean, y_pre_train_std, test_x, test_id, test_y, log_y = type2function(
    pre_train, test, elevator_exist_str, water_str, electricity_str, heat_str, house_type_str, label_str, y_str, direction_str,furnish_str,main_index,silent
)

print(f"\n=== 数据预处理完成 ===")
print(f"训练集: {X_pre_train.shape}, 测试集: {test_x.shape}")
print(f"log_y: {log_y}, y_mean: {y_pre_train_mean:.4f}, y_std: {y_pre_train_std:.4f}")

# 3. 定义反标准化函数
def inverse_transform(y_standardized, mean, std, log_y):
    """将标准化值转换回原始尺度"""
    y = y_standardized * std + mean
    if log_y:
        y = np.exp(y)
    return y

# 4. 定义性能评估函数
def evaluate_model_performance(model, X_train, y_train_std, X_test, y_test_original, 
                              mean, std, log_y, model_name):
    """评估模型性能并返回MAE和RMAE"""
    
    # 训练集预测和反标准化
    y_train_pred_std = model.predict(X_train)
    y_train_true = inverse_transform(y_train_std, mean, std, log_y)
    y_train_pred = inverse_transform(y_train_pred_std, mean, std, log_y)
    
    # 测试集预测和反标准化
    y_test_pred_std = model.predict(X_test)
    y_test_pred = inverse_transform(y_test_pred_std, mean, std, log_y)
    y_test_true = y_test_original
    
    # 计算MAE
    train_mae = mean_absolute_error(y_train_true, y_train_pred)
    test_mae = mean_absolute_error(y_test_true, y_test_pred)
    
    # 计算RMAE (MAE / mean(|y_true|))
    train_rmae = train_mae / np.mean(np.abs(y_train_true))
    test_rmae = test_mae / np.mean(np.abs(y_test_true))
    
    # 交叉验证性能
    cv_scores = cross_val_score(model, X_train, y_train_std, 
                               cv=6, scoring='neg_mean_absolute_error')
    cv_mae = -np.mean(cv_scores)
    
    # 交叉验证的RMAE估计
    y_cv_pred_std = cross_val_predict(model, X_train, y_train_std, cv=6)
    y_cv_pred = inverse_transform(y_cv_pred_std, mean, std, log_y)
    y_cv_true = inverse_transform(y_train_std, mean, std, log_y)
    cv_rmae = mean_absolute_error(y_cv_true, y_cv_pred) / np.mean(np.abs(y_cv_true))
    
    print(f"\n{model_name} 性能:")
    print(f"  训练集 MAE: {train_mae:.2f}, RMAE: {train_rmae:.4f}")
    print(f"  测试集 MAE: {test_mae:.2f}, RMAE: {test_rmae:.4f}")
    print(f"  交叉验证 RMAE: {cv_rmae:.4f}")
    
    return {
        'train_mae': train_mae,
        'train_rmae': train_rmae,
        'test_mae': test_mae,
        'test_rmae': test_rmae,
        'cv_rmae': cv_rmae
    }

# 5. train model
print(f"\n=== 开始模型训练和调优 ===")

# 5.1 OLS模型
print("训练OLS模型...")
ols = LinearRegression()
ols.fit(X_pre_train, y_pre_train)
ols_best_params = "No hyperparameters (baseline)"

# 5.2 Lasso回归
print("调优Lasso模型...")
lasso = Lasso(random_state=random_seed, max_iter=10000)
lasso_param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
lasso_grid = GridSearchCV(
    lasso, lasso_param_grid, 
    cv=6,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
lasso_grid.fit(X_pre_train, y_pre_train)
best_lasso = lasso_grid.best_estimator_
lasso_best_params = lasso_grid.best_params_

# 5.3 岭回归
print("调优Ridge模型...")
ridge = Ridge(random_state=random_seed)
ridge_param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(
    ridge, ridge_param_grid,
    cv=6,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
ridge_grid.fit(X_pre_train, y_pre_train)
best_ridge = ridge_grid.best_estimator_
ridge_best_params = ridge_grid.best_params_

# 5.4 ElasticNet
print("调优ElasticNet模型...")
elastic_net = ElasticNet(random_state=random_seed, max_iter=10000)
en_param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
en_grid = GridSearchCV(
    elastic_net, en_param_grid,
    cv=6,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
en_grid.fit(X_pre_train, y_pre_train)
best_en = en_grid.best_estimator_
en_best_params = en_grid.best_params_


best_params_dict = {
    'OLS': ols_best_params,
    'LASSO': lasso_best_params,
    'Ridge': ridge_best_params,
    'ElasticNet': en_best_params
}


print(f"\n=== best params ===")
for model_name, params in best_params_dict.items():
    print(f"{model_name}: {params}")

# 6. check all model's performance


models = {
    'OLS': ols,
    'LASSO': best_lasso,
    'Ridge': best_ridge,
    'ElasticNet': best_en
}

performance_results = {}
for name, model in models.items():
    performance_results[name] = evaluate_model_performance(
        model, X_pre_train, y_pre_train, test_x, test_y,
        y_pre_train_mean, y_pre_train_std, log_y, name
    )

# 7. find best model
best_model_name = min(performance_results.items(), 
                     key=lambda x: x[1]['test_mae'])[0]
print(f"\n=== best model: {best_model_name} ===")

# 8. performance
print(f"\n=== 最终性能表格 ===")
print("| Metrics | In sample (RMAE) | Out of sample (RMAE) | Cross-validation (RMAE) | Best Hyperparameters | Kaggle Score(模拟值) |")
print("|---------|------------------|----------------------|------------------------|---------------------|--------------|")

# 模拟Kaggle分数（实际中需要提交到Kaggle平台）
kaggle_scores = {
    'OLS': 60,
    'LASSO': 61,
    'Ridge': 62,
    'ElasticNet': 61
}

for model_name in ['OLS', 'LASSO', 'Ridge', 'ElasticNet']:
    metrics = performance_results[model_name]
    params = best_params_dict[model_name]
    kaggle_score = kaggle_scores.get(model_name, 60)
    print(f"| {model_name} | {metrics['train_rmae']:.4f} | {metrics['test_rmae']:.4f} | {metrics['cv_rmae']:.4f} | {params} | {kaggle_score} |")

# best model
best_metrics = performance_results[best_model_name]
best_params = best_params_dict[best_model_name]
best_kaggle = kaggle_scores.get(best_model_name, 62)
print(f"| Best Linear Model ({best_model_name}) | {best_metrics['train_rmae']:.4f} | {best_metrics['test_rmae']:.4f} | {best_metrics['cv_rmae']:.4f} | {best_params} | {best_kaggle} |")



原始数据形状: (96246, 46)
预训练集形状: (76996, 46)
测试集形状: (19250, 46)
knn_fill_categorical completed.

Warning: Column 'current_floor' has 4 missing in training, 1 in exam
Warning: Column '装修' has 57051 missing in training, 14214 in exam
  - 供热费: Train(70.0%) Exam(70.2%)
Target 'Price' log-transformed: skew 5.2520 -> 0.3783
  Used OLS regression: y = 0.0003 * x + 5.8904
  R² score on training data: 0.1628
Filling 建筑年代 with average method...
  Used numeric mean value from data: 2006.125855
Filling 房屋总数 with average method...
  Used numeric mean value from data: 2050.100402
Filling 楼栋总数 with average method...
  Used numeric mean value from data: 2.569134
Filling 绿 化 率 with average method...
  Used numeric mean value from data: 3.438835
Filling 容 积 率 with average method...
  Used numeric mean value from data: 0.979470
Filling 物 业 费 with average method...
  Used numeric mean value from data: 0.738400
Filling 燃气费 with average method...
  Used numeric mean value from data: 2.884106
Filling total_floo

## Regression: use all data

### preparation

In [47]:
data_x,data_y,data_y_mean,data_y_std,exam_x,exam_id,exam_y,log_y = type2function(data,exam,elevator_exist_str,water_str,electricity_str,heat_str,house_type_str,label_str,y_str,direction_str,furnish_str,main_index,silent)

knn_fill_categorical completed.

Warning: Column 'current_floor' has 5 missing in training, 0 in exam
Warning: Column '装修' has 71265 missing in training, 2682 in exam
  - 供热费: Train(70.0%) Exam(72.2%)
Target 'Price' log-transformed: skew 5.2408 -> 0.3813
  Used OLS regression: y = 0.0003 * x + 5.8907
  R² score on training data: 0.1638
Filling 建筑年代 with average method...
  Used numeric mean value from data: 2006.125945
Filling 房屋总数 with average method...
  Used numeric mean value from data: 2051.735478
Filling 楼栋总数 with average method...
  Used numeric mean value from data: 2.566681
Filling 绿 化 率 with average method...
  Used numeric mean value from data: 3.437885
Filling 容 积 率 with average method...
  Used numeric mean value from data: 0.980449
Filling 物 业 费 with average method...
  Used numeric mean value from data: 0.738799
Filling 燃气费 with average method...
  Used numeric mean value from data: 2.883715
Filling total_floor with average method...
  Used numeric mean value from data

### OLS

In [48]:
from sklearn.linear_model import LinearRegression
print(main_index)
# 假设 log_y 是布尔值，表示 y 是否被 log-transform
# y_std, y_mean 已在标准化过程中计算
# x, y, exam_final 已准备好

model = LinearRegression(fit_intercept=True)
model.fit(data_x, data_y)

# 系数与截距
beta = model.coef_
intercept = model.intercept_

# predict_data
y_pred = model.predict(exam_x)

y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

import pandas as pd
y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_OLS.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_OLS.csv", index=False, mode='a', header=False)


1


### LASSO

In [49]:
# LASSO模型
from sklearn.linear_model import Lasso
print(main_index)
# 使用字典中的最佳参数
lasso_params = best_params_dict['LASSO']
model = Lasso(**lasso_params, random_state=111, max_iter=10000)
model.fit(data_x, data_y)

# 预测
y_pred = model.predict(exam_x)
y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

import pandas as pd
y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_LASSO.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_LASSO.csv", index=False, mode='a', header=False)

1


### Ridge

In [50]:
# Ridge模型
from sklearn.linear_model import Ridge
print(main_index)
# 使用字典中的最佳参数
ridge_params = best_params_dict['Ridge']
model = Ridge(**ridge_params, random_state=111)
model.fit(data_x, data_y)

# 预测
y_pred = model.predict(exam_x)
y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_Ridge.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_Ridge.csv", index=False, mode='a', header=False)

1


### ElasticNet

In [51]:
# ElasticNet模型
print(main_index)
from sklearn.linear_model import ElasticNet

# 使用字典中的最佳参数
en_params = best_params_dict['ElasticNet']
model = ElasticNet(**en_params, random_state=111, max_iter=10000)
model.fit(data_x, data_y)

# 预测
y_pred = model.predict(exam_x)
y_pred_unscaled = y_pred * data_y_std + data_y_mean

# 如果 y 做了 log-transform，反对数
if log_y:
    y_result = np.exp(y_pred_unscaled)
else:
    y_result = y_pred_unscaled

y_result[y_result<0] = 0
# 构建 DataFrame
df_out = pd.DataFrame({
    "ID": exam_id,
    "Price": y_result
})

# 根据 main_index 选择模式
if main_index == 0:
    # 写入新文件，写表头
    df_out.to_csv("midterm_exam_ElasticNet.csv", index=False, mode='w', header=True)
else:
    # 追加到已有文件，不写表头
    df_out.to_csv("midterm_exam_ElasticNet.csv", index=False, mode='a', header=False)

1
